In [ ]:
#@title ARC-AGI-2 P147 Hidden-Parity Autolearning Calibration Lab { display-mode: "form" }
#@markdown ### 1. Identidade do experimento
EXPERIMENT_ID = "CAL-P147-hidden-parity-100-task-autolearning" #@param {type:"string"}
EXPERIMENT_NOTE = "P147 blinda ordem FinOps, subprocessos, traces e falhas do autolearning" #@param {type:"string"}
RUN_ID_SUFFIX = "" #@param {type:"string"}
#@markdown ### 1b. Protocolo supervisionado sem leakage
PILOT_TASKS = 100 #@param {type:"integer"}
EPISODES_PER_TASK = 4 #@param {type:"integer"}
OUTER_FOLDS = 5 #@param {type:"integer"}
AUTOLEARN_SEED = 20260722 #@param {type:"integer"}
BOOTSTRAP_SAMPLES = 2000 #@param {type:"integer"}
ENABLE_FULL_PROCESS_TRACE = True
STOP_ON_AUTOLEARN_FAILURE = True
#@markdown ### 2. Bundle e subset
BUNDLE_NAME = 'arc_agi2_autolearning_bundle_p147_20260801.zip' #@param {type:"string"}
TRY_DRIVE_MOUNT = True #@param {type:"boolean"}
RUN_KEYS = "0934a4d8,36a08778,981571dc,aa4ec2a5,e8686506,7666fa5d,135a2760,80a900e0,2c181942,9aaea919,20270e3b,9385bd28,269e22fb,4c7dc4dd,d8e07eb2,d35bdbdc" #@param {type:"string"}
MAX_TASKS = 105 #@param {type:"integer"}
SECONDS_PER_PROFILE_MINUTES = 420 #@param {type:"integer"}
#@markdown ### 3. Perfis Qwen
PROFILE_PRESET = "canonical_only" #@param ["canonical_only", "baseline_only", "baseline_plus_diverse", "baseline_plus_diverse_deep", "baseline_plus_deep", "custom"]
CUSTOM_PROFILES = "koushik_plus,koushik_diverse,koushik_deep" #@param {type:"string"}
#@markdown ### 3b. Matriz de geração. O preset recomendado altera somente as sementes.
PORTFOLIO_PRESET = "dual_seed_koushik" #@param ["dual_seed_koushik", "off", "custom"]
CUSTOM_RUN_MATRIX_JSON = "[]" #@param {type:"string"}
#@markdown ### 4. Seletor e gates
SELECTOR_PRESET = "kgmon" #@param ["kgmon", "submit_public_3389", "topology_second", "portfolio", "custom"]
CUSTOM_SELECTOR_WEIGHTS = "selection_mode=public_kgmon" #@param {type:"string"}
MAX_DUPLICATE_ATTEMPT_RATE = 0.15 #@param {type:"number"}
MAX_ATTEMPT2_INPUT_FALLBACK_RATE = 0.15 #@param {type:"number"}
USE_SYMBOLIC = False #@param {type:"boolean"}
MISSING_SYMBOLIC_FALLBACK = True #@param {type:"boolean"}
STOP_AFTER_BASELINE_FAILURE = True #@param {type:"boolean"}
#@markdown ### 4b. Sweep barato de selector, sem refazer inferencia
SELECTOR_SWEEP_ENABLED = True #@param {type:"boolean"}
SELECTOR_SWEEP_MODES = "public_3389,public_3389_topology_second,public_3389_portfolio_first,public_3389_vote_first,public_probmul,public_kgmon,public_portfolio,portfolio" #@param {type:"string"}
#@markdown ### 5. Overrides avancados do perfil Qwen. Deixe vazio para usar o perfil padrao.
TRAIN_AUG_N = "" #@param {type:"string"}
EVAL_AUG_N = "" #@param {type:"string"}
DFS_SECONDS = "" #@param {type:"string"}
PUZZLE_TIMEOUT_SECONDS = "" #@param {type:"string"}
MIN_START_REMAINING_SECONDS = "" #@param {type:"string"}
MAX_SCORE_PROB = "" #@param {type:"string"}
TRAIN_PRECISION = "auto" #@param ["auto", "bf16", "fp16", "fp32"]
#@markdown ### 6. Runtime e staging
FORCE_GPU_COUNT = "1" #@param ["1", "2", "4"] {allow-input: true}
REQUIRE_L4_TIMING = False #@param {type:"boolean"}
INSTALL_COMPAT_UNSLOTH = "auto"
STRICT_FLASH_CAUSAL = False #@param {type:"boolean"}
#@markdown ### 7. Logs
HF_LOG_ENABLED_FORM = True #@param {type:"boolean"}
HF_LOG_DATASET_FORM = "" #@param {type:"string"}
HF_LOG_SYNC_SECONDS_FORM = 180 #@param {type:"integer"}
DRIVE_LOG_ROOT_FORM = "/content/drive/MyDrive/arc2016_colab_live_logs" #@param {type:"string"}
DRIVE_LOG_SYNC_SECONDS_FORM = 30 #@param {type:"integer"}
#@markdown ---
#@markdown **Regra:** este notebook coleta evidencia e nunca submete ao Kaggle.

import json
import base64
from collections import deque
import io
import hashlib
import importlib
import logging
import os
from pathlib import Path, PurePosixPath
import re
import signal
import shutil
import shlex
import subprocess
import tempfile
import sys
import threading
import time
import traceback
import warnings
import zipfile


BOOTSTRAP_JOURNAL_PATH = Path("/content/arc_p147_bootstrap.jsonl")
BOOTSTRAP_SENSITIVE_ENV_NAMES = {
    "HF_TOKEN", "HF_KEY", "HUGGING_FACE_HUB_TOKEN", "OPENROUTER_API_KEY",
    "KAGGLE_USERNAME", "KAGGLE_KEY", "GH_TOKEN", "GITHUB_TOKEN",
}


def _redact_bootstrap_text(text: str) -> str:
    redacted = str(text)
    for name in BOOTSTRAP_SENSITIVE_ENV_NAMES:
        value = os.environ.get(name)
        if value and len(value) >= 6:
            redacted = redacted.replace(value, f"<{name}:redacted>")
    return redacted


def bootstrap_event(event: str, payload: dict | None = None) -> None:
    record = {
        "ts_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "event": event,
        "payload": payload or {},
    }
    serialized = _redact_bootstrap_text(json.dumps(record, ensure_ascii=True, sort_keys=True))
    with open(BOOTSTRAP_JOURNAL_PATH, "a", encoding="utf-8", newline="\n") as handle:
        handle.write(serialized + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def _bootstrap_excepthook(exc_type, exc, tb):
    try:
        bootstrap_event("bootstrap_uncaught_exception", {
            "type": exc_type.__name__,
            "error": str(exc)[:2000],
            "traceback": "".join(traceback.format_exception(exc_type, exc, tb))[-12000:],
        })
    finally:
        sys.__excepthook__(exc_type, exc, tb)


def _bootstrap_ipython_failure_handler(self, etype, value, tb, tb_offset=None):
    bootstrap_event("bootstrap_ipython_exception", {
        "type": getattr(etype, "__name__", str(etype)),
        "error": repr(value)[:2000],
        "traceback": "".join(traceback.format_exception(etype, value, tb))[-12000:],
    })
    return traceback.format_exception(etype, value, tb)


sys.excepthook = _bootstrap_excepthook
try:
    _bootstrap_ipython = get_ipython()
except NameError:
    _bootstrap_ipython = None
if _bootstrap_ipython is not None:
    _bootstrap_ipython.set_custom_exc((BaseException,), _bootstrap_ipython_failure_handler)
    bootstrap_event("bootstrap_ipython_handler_installed")
else:
    bootstrap_event("bootstrap_ipython_handler_unavailable")
bootstrap_event("bootstrap_started", {"python": sys.version})


ROOT_DIR = "/content/arc_agi2_autolearning_p147"
LAB_CONFIG_VERSION = "p147-line-hardened-fail-fast-autolearning"
EMBEDDED_BUNDLE_B64 = '...'
EMBEDDED_BUNDLE_SHA256 = '8a16a5867515a36e3aedb7e64d01d4f554147e85dc622f01e1b4a379d0aa8d1c'
COLAB_RELEASE_POLICY = (
    "Never update an opened Colab notebook file in place. "
    "Each release gets a new Drive file ID and a versioned bundle."
)
P147_ATOMIC_SPECS = [
    "kaggle==2.2.3",
    "unsloth==2025.9.7",
    "unsloth_zoo==2025.9.9",
    "transformers==4.55.4",
    "peft==0.18.1",
    "datasets==3.6.0",
    "accelerate==1.13.0",
    "trl==0.22.2",
    "bitsandbytes==0.48.2",
    "xformers==0.0.35",
    "huggingface_hub==0.36.2",
    "tokenizers==0.21.4",
    "scikit-learn==1.7.2",
    "torchao==0.17.0",
]
COLAB_COMPAT_UNSLOTH_SPEC = " ".join(P147_ATOMIC_SPECS)
P147_EXPECTED_EXACT = {
    "kaggle": "2.2.3",
    "unsloth": "2025.9.7",
    "unsloth_zoo": "2025.9.9",
    "transformers": "4.55.4",
    "peft": "0.18.1",
    "datasets": "3.6.0",
    "accelerate": "1.13.0",
    "trl": "0.22.2",
    "bitsandbytes": "0.48.2",
    "xformers": "0.0.35",
    "huggingface-hub": "0.36.2",
    "tokenizers": "0.21.4",
    "scikit-learn": "1.7.2",
    "torchao": "0.17.0",
}
P147_EXPECTED_RUNTIME_PUBLIC = {
    "torch": "2.11.0",
    "torchvision": "0.26.0",
    "triton": "3.6.0",
}
DEPENDENCY_CONTRACT_PATH = Path(ROOT_DIR) / "dependency_contract_p147.json"
FLASH_CAUSAL_STRICT = os.environ.get(
    "ARC_COLAB_STRICT_FLASH_CAUSAL",
    "1" if bool(STRICT_FLASH_CAUSAL) else "0",
).strip()

def csv_items(text):
    return [part.strip() for part in str(text or "").split(",") if part.strip()]


if Path(BUNDLE_NAME).name != BUNDLE_NAME or not str(BUNDLE_NAME).endswith(".zip"):
    raise ValueError("BUNDLE_NAME must be a plain .zip filename, not a path")

RUN_ID_SUFFIX = re.sub(r"[^0-9A-Za-z_.-]+", "-", str(RUN_ID_SUFFIX or "").strip()).strip("-_.")[:60]

RUN_KEYS_LIST = csv_items(RUN_KEYS)
if not RUN_KEYS_LIST:
    raise ValueError("RUN_KEYS cannot be empty")
invalid_run_keys = [key for key in RUN_KEYS_LIST if not re.fullmatch(r"[0-9a-f]{8}", key)]
if invalid_run_keys:
    raise ValueError(f"RUN_KEYS has invalid ARC task ids: {invalid_run_keys}")
if len(RUN_KEYS_LIST) != len(set(RUN_KEYS_LIST)):
    raise ValueError("RUN_KEYS must not contain duplicates")
RUN_KEYS = ",".join(RUN_KEYS_LIST)
CONFIGURED_RUN_KEYS_LIST = tuple(RUN_KEYS_LIST)
CONFIGURED_RUN_KEYS = RUN_KEYS
EFFECTIVE_LOPO_KEYS_LIST = ()
EFFECTIVE_LOPO_KEYS = ""

MAX_TASKS = int(MAX_TASKS)
if not 1 <= MAX_TASKS <= 1000:
    raise ValueError("MAX_TASKS must be between 1 and 1000 for this lab notebook")

SECONDS_PER_PROFILE_MINUTES = int(SECONDS_PER_PROFILE_MINUTES)
if not 5 <= SECONDS_PER_PROFILE_MINUTES <= 600:
    raise ValueError("SECONDS_PER_PROFILE_MINUTES must be between 5 and 600")

PROFILE_PRESETS = {
    "canonical_only": ["koushik"],
    "baseline_plus_diverse_deep": ["koushik_plus", "koushik_diverse", "koushik_deep"],
    "baseline_only": ["koushik_plus"],
    "baseline_plus_deep": ["koushik_plus", "koushik_deep"],
    "baseline_plus_diverse": ["koushik_plus", "koushik_diverse"],
}
PROFILES = csv_items(CUSTOM_PROFILES) if PROFILE_PRESET == "custom" else PROFILE_PRESETS.get(PROFILE_PRESET, [])
if not PROFILES or PROFILES[0] not in {"koushik", "koushik_plus"}:
    raise ValueError("PROFILES must start with baseline profile 'koushik' or 'koushik_plus'")
if any(not re.fullmatch(r"[0-9A-Za-z_.-]+", profile) for profile in PROFILES):
    raise ValueError(f"PROFILES contains unsafe profile names: {PROFILES}")
if len(PROFILES) != len(set(PROFILES)):
    raise ValueError("PROFILES must not contain duplicates")

DUAL_SEED_RUN_MATRIX = [
    {
        "tag": "seed-a",
        "profile": "koushik",
        "lora_rank": 256,
        "train_aug_n": 16,
        "eval_aug_n": 2,
        "dfs_seconds": 540,
        "puzzle_timeout_seconds": 1200,
        "max_score_prob": 0.2,
        "global_seed": 42,
        "peft_random_state": 42,
        "train_seed": 42,
        "train_aug_seed": 1,
        "eval_aug_seed": 2,
        "puzzle_seed_salt": "",
        "score_aug_seed_salt": "",
    },
    {
        "tag": "seed-b",
        "profile": "koushik",
        "lora_rank": 256,
        "train_aug_n": 16,
        "eval_aug_n": 2,
        "dfs_seconds": 540,
        "puzzle_timeout_seconds": 1200,
        "max_score_prob": 0.2,
        "global_seed": 314159,
        "peft_random_state": 271828,
        "train_seed": 161803,
        "train_aug_seed": 104729,
        "eval_aug_seed": 130363,
        "puzzle_seed_salt": "dual-b-puzzle",
        "score_aug_seed_salt": "dual-b-score",
    },
]
PORTFOLIO_PRESET = str(PORTFOLIO_PRESET).strip().lower()
if PORTFOLIO_PRESET == "dual_seed_koushik":
    RUN_MATRIX = DUAL_SEED_RUN_MATRIX
elif PORTFOLIO_PRESET == "off":
    RUN_MATRIX = []
elif PORTFOLIO_PRESET == "custom":
    try:
        RUN_MATRIX = json.loads(str(CUSTOM_RUN_MATRIX_JSON or "[]"))
    except json.JSONDecodeError as exc:
        raise ValueError(f"CUSTOM_RUN_MATRIX_JSON is invalid JSON: {exc.msg}") from exc
else:
    raise ValueError("PORTFOLIO_PRESET must be dual_seed_koushik, off, or custom")
if not isinstance(RUN_MATRIX, list):
    raise ValueError("RUN_MATRIX must be a JSON list")
if RUN_MATRIX and len(PROFILES) != 1:
    raise ValueError("Portfolio mode requires exactly one outer profile; choose a one-profile preset")

SELECTOR_PRESETS = {
    "kgmon": "selection_mode=public_kgmon",
    "topology_second": "selection_mode=public_3389_topology_second",
    "submit_public_3389": "selection_mode=public_3389",
    "portfolio": "selection_mode=portfolio",
}
if SELECTOR_PRESET == "custom":
    SELECTOR_WEIGHT_SPEC = CUSTOM_SELECTOR_WEIGHTS.strip()
elif SELECTOR_PRESET in SELECTOR_PRESETS:
    SELECTOR_WEIGHT_SPEC = SELECTOR_PRESETS[SELECTOR_PRESET]
else:
    raise ValueError(f"Unknown SELECTOR_PRESET={SELECTOR_PRESET!r}")
if not SELECTOR_WEIGHT_SPEC:
    raise ValueError("SELECTOR_WEIGHT_SPEC cannot be empty")
SELECTOR_SWEEP_MODES = ",".join(csv_items(SELECTOR_SWEEP_MODES))
if SELECTOR_SWEEP_ENABLED and not SELECTOR_SWEEP_MODES:
    raise ValueError("SELECTOR_SWEEP_MODES cannot be empty when SELECTOR_SWEEP_ENABLED is true")

SECONDS_PER_PROFILE = SECONDS_PER_PROFILE_MINUTES * 60
if not 0.0 <= float(MAX_DUPLICATE_ATTEMPT_RATE) <= 1.0:
    raise ValueError("MAX_DUPLICATE_ATTEMPT_RATE must be between 0 and 1")
if not 0.0 <= float(MAX_ATTEMPT2_INPUT_FALLBACK_RATE) <= 1.0:
    raise ValueError("MAX_ATTEMPT2_INPUT_FALLBACK_RATE must be between 0 and 1")
PORTFOLIO_RUN_COUNT = len(RUN_MATRIX) if RUN_MATRIX else 1
NOMINAL_SECONDS_PER_PORTFOLIO_RUN = SECONDS_PER_PROFILE / PORTFOLIO_RUN_COUNT
FROZEN_P137_REFERENCE = {
    "source": "P137 run arc2016-colab-qwen-ab-20260722T220700Z",
    "evidence_scope": "public_training_lopo_proxy_not_kaggle_score",
    "profile": "koushik",
    "portfolio_preset": "off",
    "selector_score": 0.8125,
    "oracle_score": 0.8125,
    "selector_correct_outputs": 13,
    "oracle_correct_outputs": 13,
    "outputs_total": 16,
    "returncode": 0,
    "total_budget_seconds": 25200,
}
FROZEN_P147_PROTOCOL_REFERENCE = {
    "source_challenges_sha256": "f8454239fb634f74a14f2a780c9fd44762d08ab70b9584f836eda3ff75695518",
    "source_solutions_sha256": "cdf36bf5c02601a60efa5036ebf64d56aa18cdd4b82366a67ce76bc8a33ff928",
    "episode_challenges_sha256": "d3aff2961c7d18f12f9e6c50cc9bdd0c68fc30bce01c7360d2d884a44535bf56",
    "episode_solutions_sha256": "86e225e7940a28e5d482361493b782391276a184e64e419989171b0b23921d94",
    "episode_protocol": "original_test",
    "source_partition": "official_training",
    "task_count": 100,
    "episode_count": 105,
    "available_holdout_count": 105,
    "all_available_holdouts_included": True,
    "hidden_test_information_parity": True,
}
FROZEN_REFERENCE = FROZEN_P147_PROTOCOL_REFERENCE
EPISODE_PROTOCOL = FROZEN_REFERENCE["episode_protocol"]
if not isinstance(EPISODE_PROTOCOL, str) or not EPISODE_PROTOCOL:
    raise RuntimeError("P147 frozen episode_protocol must be a non-empty string")
if SECONDS_PER_PROFILE != FROZEN_P137_REFERENCE["total_budget_seconds"]:
    raise ValueError("P147 must preserve the P137 total generator budget for a comparable portfolio-policy test")
FORCE_GPU_COUNT = str(FORCE_GPU_COUNT).strip()
if FORCE_GPU_COUNT not in {"1", "2", "4"}:
    raise ValueError("FORCE_GPU_COUNT must be one of 1, 2, 4")
INSTALL_COMPAT_UNSLOTH = str(INSTALL_COMPAT_UNSLOTH).strip().lower()
if INSTALL_COMPAT_UNSLOTH not in {"auto", "force", "skip"}:
    raise ValueError("INSTALL_COMPAT_UNSLOTH must be auto, force, or skip")
HF_LOG_SYNC_SECONDS = int(HF_LOG_SYNC_SECONDS_FORM)
if not 15 <= HF_LOG_SYNC_SECONDS <= 600:
    raise ValueError("HF_LOG_SYNC_SECONDS_FORM must be between 15 and 600")
DRIVE_LOG_SYNC_SECONDS = int(DRIVE_LOG_SYNC_SECONDS_FORM)
if not 10 <= DRIVE_LOG_SYNC_SECONDS <= 600:
    raise ValueError("DRIVE_LOG_SYNC_SECONDS_FORM must be between 10 and 600")
DRIVE_LOG_ROOT = str(DRIVE_LOG_ROOT_FORM or "").strip() or "/content/drive/MyDrive/arc2016_colab_live_logs"

QWEN_OPTIONAL_OVERRIDES = {
    "ARC_QWEN_TRAIN_AUG_N": TRAIN_AUG_N,
    "ARC_QWEN_EVAL_AUG_N": EVAL_AUG_N,
    "ARC_QWEN_DFS_SECONDS": DFS_SECONDS,
    "ARC_QWEN_PUZZLE_TIMEOUT_SECONDS": PUZZLE_TIMEOUT_SECONDS,
    "ARC_QWEN_MIN_START_REMAINING_SECONDS": MIN_START_REMAINING_SECONDS,
    "ARC_QWEN_MAX_SCORE_PROB": MAX_SCORE_PROB,
    "ARC_QWEN_TRAIN_PRECISION": TRAIN_PRECISION,
}
QWEN_OPTIONAL_OVERRIDES = {
    key: str(value).strip()
    for key, value in QWEN_OPTIONAL_OVERRIDES.items()
    if str(value).strip()
}
if RUN_MATRIX:
    QWEN_OPTIONAL_OVERRIDES["ARC_QWEN_RUN_MATRIX_JSON"] = json.dumps(
        RUN_MATRIX, separators=(",", ":"), sort_keys=True
    )
    QWEN_OPTIONAL_OVERRIDES["ARC_QWEN_PORTFOLIO_CONTINUE_ON_ERROR"] = "0"

# Keep Kaggle kernel-output staging bounded. This mirrors
# hf_stage_kaggle_assets.DEFAULT_KAGGLE_OUTPUT_PATTERN and also protects reruns
# that accidentally use an older bundle where the default was not applied.
KAGGLE_OUTPUT_FILE_PATTERN = (
    r"^(unsloth|unsloth_zoo|trl|bitsandbytes|flash_attn|cut_cross_entropy|"
    r"xformers|triton|tyro|shtab|docstring_parser)(/|-)"
)

# HF logging. Leave ARC_HF_LOG_DATASET empty to auto-create/use
# <hf-username>/arc-2016-colab-logs as a private dataset.
HF_LOG_ENABLED = bool(HF_LOG_ENABLED_FORM) and os.environ.get("ARC_HF_LOG_ENABLED", "1").lower() not in {"0", "false", "no"}
HF_LOG_DATASET = os.environ.get("ARC_HF_LOG_DATASET") or str(HF_LOG_DATASET_FORM or "").strip()
RUN_ID_BASE = f"arc2016-colab-qwen-ab-{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}-{time.time_ns() % 1_000_000_000:09d}"
runtime_run_nonce = (
    time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
    + f"-{time.time_ns() % 1_000_000_000:09d}"
)
RUN_ID = f"{RUN_ID_BASE}-{runtime_run_nonce}"
if RUN_ID_SUFFIX:
    RUN_ID = f"{RUN_ID}-{RUN_ID_SUFFIX}"
if re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,127}", RUN_ID) is None:
    raise ValueError(f"unsafe generated run id: {RUN_ID!r}")
P147_EXECUTABLE_SOURCE_SHA256 = 'a76e0dfe0b1af30280271a4b65aca43bed7190ff31804706dca568a33c3f4872'
P147_BUILDER_SHA256 = '24d4dff960377f10f12a273a0f4c385e343483ab412b712d2b58f4f848c09025'
HF_BRIDGE = None
DRIVE_LOG_MIRROR = None

# Keep logs focused on actionable events. These filters only silence known,
# non-critical notebook/HF noise; exceptions and command failures still surface.
QUIET_ENV_DEFAULTS = {
    "TF_CPP_MIN_LOG_LEVEL": "3",
    "TF_ENABLE_ONEDNN_OPTS": "0",
    "USE_TF": "0",
    "USE_FLAX": "0",
    "TOKENIZERS_PARALLELISM": "false",
}
for key, value in QUIET_ENV_DEFAULTS.items():
    os.environ.setdefault(key, value)

NONCRITICAL_LOG_PATTERNS = [
    re.compile(r"WARNING: unsloth .* does not provide the extra 'triton'"),
    re.compile(r".*tensorflow/core/util/port\.cc:.*oneDNN custom operations are on.*"),
    re.compile(r".*tensorflow/core/platform/cpu_feature_guard\.cc:.*optimized to use available CPU instructions.*"),
    re.compile(r"To enable the following instructions: .*"),
    re.compile(r"Flax classes are deprecated and will be removed in Diffusers.*"),
    re.compile(r".*UserWarning: Unsloth fused-forward install skipped: requires transformers >= 4\.56\.0\..*"),
    re.compile(r"\s*_install_fused_forward\(\)\s*$"),
]
warnings.filterwarnings("ignore", message=r".*No files have been modified since last commit.*")
warnings.filterwarnings("ignore", message=r".*resume_download.*deprecated.*")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)


def is_noncritical_log_line(line: str) -> bool:
    return any(pattern.match(line.rstrip()) for pattern in NONCRITICAL_LOG_PATTERNS)


def section(title: str) -> None:
    print("\n" + "=" * 88)
    print(title)
    print("=" * 88)
    if HF_BRIDGE is not None:
        HF_BRIDGE.event("section", {"title": title})


def atomic_write_text(path: Path, text: str, *, encoding: str = "utf-8") -> None:
    path = Path(path)
    sanitizer = globals().get("_redact_sensitive_text")
    safe_text = sanitizer(text) if callable(sanitizer) else text
    firewall = globals().get("assert_no_sensitive_bytes")
    if callable(firewall):
        firewall(safe_text.encode(encoding), context=f"atomic text write {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + f".tmp.{os.getpid()}.{threading.get_ident()}")
    try:
        with open(temp, "w", encoding=encoding, newline="\n") as handle:
            handle.write(safe_text)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)


def atomic_write_bytes(path: Path, payload: bytes) -> None:
    path = Path(path)
    if not isinstance(payload, bytes):
        raise TypeError(f"payload must be bytes: path={path}")
    firewall = globals().get("assert_no_sensitive_bytes")
    if callable(firewall):
        firewall(payload, context=f"atomic bytes write {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + f".tmp.{os.getpid()}.{threading.get_ident()}")
    try:
        with open(temp, "wb") as handle:
            handle.write(payload)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_private_json(path: Path, payload: dict) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True, mode=0o700)
    os.chmod(path.parent, 0o700)
    temp = path.with_name(path.name + f".tmp.{os.getpid()}.{threading.get_ident()}")
    descriptor = None
    try:
        descriptor = os.open(temp, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="\n") as handle:
            descriptor = None
            json.dump(payload, handle, ensure_ascii=True, separators=(",", ":"), sort_keys=True)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temp, path)
        os.chmod(path, 0o600)
    finally:
        if descriptor is not None:
            os.close(descriptor)
        temp.unlink(missing_ok=True)


LAB_PARAMETERS = {
    "lab_config_version": LAB_CONFIG_VERSION,
    "experiment_id": EXPERIMENT_ID,
    "experiment_note": EXPERIMENT_NOTE,
    "run_id_suffix": RUN_ID_SUFFIX,
    "run_id": RUN_ID,
    "pilot_tasks": int(PILOT_TASKS),
    "episodes_per_task": int(EPISODES_PER_TASK),
    "outer_folds": int(OUTER_FOLDS),
    "autolearn_seed": int(AUTOLEARN_SEED),
    "bootstrap_samples": int(BOOTSTRAP_SAMPLES),
    "enable_full_process_trace": bool(ENABLE_FULL_PROCESS_TRACE),
    "stop_on_autolearn_failure": bool(STOP_ON_AUTOLEARN_FAILURE),
    "bundle_name": BUNDLE_NAME,
    "embedded_bundle_sha256": EMBEDDED_BUNDLE_SHA256,
    "try_drive_mount": bool(TRY_DRIVE_MOUNT),
    "configured_run_keys": CONFIGURED_RUN_KEYS,
    "configured_run_key_count": len(CONFIGURED_RUN_KEYS_LIST),
    "effective_lopo_keys": EFFECTIVE_LOPO_KEYS,
    "effective_lopo_key_count": len(EFFECTIVE_LOPO_KEYS_LIST),
    "max_tasks": int(MAX_TASKS),
    "seconds_per_profile": int(SECONDS_PER_PROFILE),
    "profiles": PROFILES,
    "profile_preset": PROFILE_PRESET,
    "portfolio_preset": PORTFOLIO_PRESET,
    "run_matrix": RUN_MATRIX,
    "portfolio_run_count": PORTFOLIO_RUN_COUNT,
    "nominal_seconds_per_portfolio_run": NOMINAL_SECONDS_PER_PORTFOLIO_RUN,
    "frozen_p137_reference": FROZEN_P137_REFERENCE,
    "frozen_p147_protocol_reference": FROZEN_P147_PROTOCOL_REFERENCE,
    "selector_preset": SELECTOR_PRESET,
    "selector_weight_spec": SELECTOR_WEIGHT_SPEC,
    "selector_sweep_enabled": bool(SELECTOR_SWEEP_ENABLED),
    "selector_sweep_modes": SELECTOR_SWEEP_MODES,
    "max_duplicate_attempt_rate": float(MAX_DUPLICATE_ATTEMPT_RATE),
    "max_attempt2_input_fallback_rate": float(MAX_ATTEMPT2_INPUT_FALLBACK_RATE),
    "use_symbolic": bool(USE_SYMBOLIC),
    "missing_symbolic_fallback": bool(MISSING_SYMBOLIC_FALLBACK),
    "stop_after_baseline_failure": bool(STOP_AFTER_BASELINE_FAILURE),
    "qwen_optional_overrides": QWEN_OPTIONAL_OVERRIDES,
    "force_gpu_count": str(FORCE_GPU_COUNT),
    "require_l4_timing": bool(REQUIRE_L4_TIMING),
    "strict_flash_causal": FLASH_CAUSAL_STRICT,
    "install_compat_unsloth": INSTALL_COMPAT_UNSLOTH,
    "hf_log_enabled": bool(HF_LOG_ENABLED),
    "hf_log_dataset": HF_LOG_DATASET,
    "hf_log_sync_seconds": int(HF_LOG_SYNC_SECONDS),
    "drive_log_root": DRIVE_LOG_ROOT,
    "drive_log_sync_seconds": int(DRIVE_LOG_SYNC_SECONDS),
}


def seal_lab_parameters() -> str:
    canonical = json.dumps(
        {key: value for key, value in LAB_PARAMETERS.items() if key != "lab_parameters_sha256"},
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    ).encode("utf-8")
    digest = hashlib.sha256(canonical).hexdigest()
    LAB_PARAMETERS["lab_parameters_sha256"] = digest
    return digest


LAB_PARAMETERS_SHA256 = seal_lab_parameters()


def runtime_resource_snapshot() -> dict:
    snapshot = {}
    try:
        usage = shutil.disk_usage("/content")
        snapshot.update({
            "disk_total_bytes": usage.total,
            "disk_used_bytes": usage.used,
            "disk_free_bytes": usage.free,
        })
    except Exception as exc:
        snapshot["disk_probe_error"] = f"{type(exc).__name__}: {exc}"[:240]
    try:
        meminfo = {}
        for line in Path("/proc/meminfo").read_text(encoding="ascii", errors="strict").splitlines():
            name, value = line.split(":", 1)
            meminfo[name] = int(value.strip().split()[0]) * 1024
        snapshot["host_mem_total_bytes"] = meminfo.get("MemTotal")
        snapshot["host_mem_available_bytes"] = meminfo.get("MemAvailable")
    except Exception as exc:
        snapshot["host_memory_probe_error"] = f"{type(exc).__name__}: {exc}"[:240]
    try:
        process_status = {}
        for line in Path("/proc/self/status").read_text(encoding="ascii", errors="strict").splitlines():
            if ":" in line:
                name, value = line.split(":", 1)
                process_status[name] = value.strip()
        snapshot["process_vm_rss"] = process_status.get("VmRSS")
        snapshot["process_vm_hwm"] = process_status.get("VmHWM")
    except Exception as exc:
        snapshot["process_memory_probe_error"] = f"{type(exc).__name__}: {exc}"[:240]
    try:
        torch_module = sys.modules.get("torch")
        if torch_module is not None and torch_module.cuda.is_available():
            snapshot.update({
                "torch_cuda_allocated_bytes": torch_module.cuda.memory_allocated(),
                "torch_cuda_reserved_bytes": torch_module.cuda.memory_reserved(),
                "torch_cuda_max_allocated_bytes": torch_module.cuda.max_memory_allocated(),
                "torch_cuda_max_reserved_bytes": torch_module.cuda.max_memory_reserved(),
            })
    except Exception as exc:
        snapshot["torch_cuda_memory_probe_error"] = f"{type(exc).__name__}: {exc}"[:240]
    try:
        probe = subprocess.run(
            ["/usr/bin/nvidia-smi", "--query-gpu=index,uuid,memory.used,memory.total,utilization.gpu", "--format=csv,noheader,nounits"],
            text=True, capture_output=True, check=False, timeout=5,
        )
        snapshot["nvidia_smi_returncode"] = probe.returncode
        snapshot["gpu_resource_rows"] = [line.strip() for line in probe.stdout.splitlines() if line.strip()]
    except Exception as exc:
        snapshot["gpu_probe_error"] = f"{type(exc).__name__}: {exc}"[:240]
    try:
        process_probe = subprocess.run(
            ["/usr/bin/nvidia-smi", "--query-compute-apps=pid,used_gpu_memory", "--format=csv,noheader,nounits"],
            text=True, capture_output=True, check=False, timeout=5,
        )
        snapshot["gpu_process_probe_returncode"] = process_probe.returncode
        snapshot["gpu_process_rows"] = [line.strip() for line in process_probe.stdout.splitlines() if line.strip()]
    except Exception as exc:
        snapshot["gpu_process_probe_error"] = f"{type(exc).__name__}: {exc}"[:240]
    return snapshot


class TeeStream:
    def __init__(self, stream, path: Path):
        self.stream = stream
        self.path = path
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.file = open(path, "a", encoding="utf-8", buffering=1)
        self._lock = threading.RLock()

    def write(self, data):
        with self._lock:
            safe_data = _redact_sensitive_text(data)
            self.stream.write(safe_data)
            self.file.write(safe_data)
        return len(data)

    def flush(self):
        with self._lock:
            self.stream.flush()
            self.file.flush()

    def close(self):
        with self._lock:
            if not self.file.closed:
                self.file.flush()
                self.file.close()

    def __getattr__(self, name):
        return getattr(self.stream, name)


class HFLogBridge:
    def __init__(self, *, token: str, run_id: str, dataset_repo: str, sync_seconds: int):
        self.token = token
        self.run_id = run_id
        self.sync_seconds = max(15, int(sync_seconds))
        self.log_dir = Path("/content/arc2016_hf_logs") / run_id
        self.log_dir.mkdir(parents=True, exist_ok=False)
        self.stdout_path = self.log_dir / "stdout.log"
        self.stderr_path = self.log_dir / "stderr.log"
        self.events_path = self.log_dir / "events.jsonl"
        self.heartbeat_path = self.log_dir / "heartbeat.json"
        self.summary_path = self.log_dir / "run_summary.json"
        self.artifact_index_path = self.log_dir / "artifact_upload_index.json"
        self.bootstrap_path = self.log_dir / "bootstrap.jsonl"
        if BOOTSTRAP_JOURNAL_PATH.is_file():
            shutil.copy2(BOOTSTRAP_JOURNAL_PATH, self.bootstrap_path)
        self.enabled = False
        self.repo_id = dataset_repo
        self.repo_url = None
        self.api = None
        self._stop = threading.Event()
        self._lock = threading.RLock()
        self._sync_lock = threading.Lock()
        self._thread = None
        self._sync_errors = 0
        self._uploaded_signatures = {}
        self._uploaded_receipts = {}
        self._runtime_state = {
            "active_phase": "initialization",
            "active_command": None,
            "last_progress_utc": None,
            "command_started_epoch": None,
        }

        if not HF_LOG_ENABLED:
            self.event("hf_logging_disabled", {"reason": "ARC_HF_LOG_ENABLED=0"}, upload=False)
            return
        if not token:
            self.event("hf_logging_disabled", {"reason": "missing HF_TOKEN/HF_KEY"}, upload=False)
            return

        try:
            try:
                from huggingface_hub import HfApi
            except Exception:
                subprocess.run(
                    [sys.executable, "-m", "pip", "--isolated", "install", "-q", "huggingface_hub>=0.34.0,<1.0"],
                    env=safe_package_install_env(),
                    check=True,
                    timeout=300,
                )
                from huggingface_hub import HfApi

            self.api = HfApi(token=token)
            who = self.api.whoami(token=token)
            username = who.get("name") or who.get("fullname") or who.get("email", "unknown").split("@")[0]
            if not self.repo_id:
                self.repo_id = f"{username}/arc-2016-colab-logs"
            self.api.create_repo(
                repo_id=self.repo_id,
                repo_type="dataset",
                private=True,
                exist_ok=True,
                token=token,
            )
            repo_info = self.api.repo_info(repo_id=self.repo_id, repo_type="dataset", token=token, timeout=30)
            if not bool(getattr(repo_info, "private", False)):
                self.api.update_repo_settings(
                    repo_id=self.repo_id, repo_type="dataset", private=True, token=token
                )
                repo_info = self.api.repo_info(repo_id=self.repo_id, repo_type="dataset", token=token, timeout=30)
            if not bool(getattr(repo_info, "private", False)):
                raise RuntimeError(f"HF log dataset is not private: {self.repo_id}")
            self.repo_url = f"https://huggingface.co/datasets/{self.repo_id}/tree/main/runs/{self.run_id}"
            self.enabled = True
            self.event("hf_logging_started", {
                "repo_id": self.repo_id,
                "repo_url": self.repo_url,
                "sync_seconds": self.sync_seconds,
            }, upload=False)
            self.write_heartbeat("started")
            self._thread = threading.Thread(target=self._loop, daemon=True)
            self._thread.start()
        except Exception as exc:
            self.enabled = False
            safe_error = _redact_sensitive_text(f"{type(exc).__name__}: {exc}")
            print(f"[hf-logging] start failed: {safe_error}", file=sys.__stderr__)
            self.event("hf_logging_start_failed", {"error": safe_error}, upload=False)

    def event(self, name: str, payload: dict | None = None, *, upload: bool = False) -> None:
        record = {
            "ts_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "run_id": self.run_id,
            "event": name,
            "payload": payload or {},
        }
        with self._lock:
            with open(self.events_path, "a", encoding="utf-8") as f:
                f.write(_redact_sensitive_text(json.dumps(record, ensure_ascii=True, sort_keys=True)) + "\n")
                f.flush()
                os.fsync(f.fileno())
        if upload:
            self.sync_once(heartbeat_status=None)

    def update_runtime_state(self, **values) -> None:
        with self._lock:
            self._runtime_state.update(values)

    def write_heartbeat(self, status: str, extra: dict | None = None) -> None:
        data = {
            "ts_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "run_id": self.run_id,
            "status": status,
            "repo_id": self.repo_id,
            "repo_url": self.repo_url,
            "sync_errors": self._sync_errors,
        }
        data.update(self._runtime_state)
        data.update(runtime_resource_snapshot())
        if extra:
            data.update(extra)
        atomic_write_text(self.heartbeat_path, json.dumps(data, ensure_ascii=True, indent=2))

    def write_summary(self, status: str, extra: dict | None = None) -> None:
        data = {
            "run_id": self.run_id,
            "status": status,
            "repo_id": self.repo_id,
            "repo_url": self.repo_url,
            "lab_config_version": LAB_CONFIG_VERSION,
            "experiment_id": EXPERIMENT_ID,
            "experiment_note": EXPERIMENT_NOTE,
            "bundle_name": BUNDLE_NAME,
            "profiles": PROFILES,
            "configured_run_keys": CONFIGURED_RUN_KEYS,
            "effective_lopo_keys": EFFECTIVE_LOPO_KEYS,
            "max_tasks": MAX_TASKS,
            "seconds_per_profile": SECONDS_PER_PROFILE,
            "force_gpu_count": FORCE_GPU_COUNT,
            "selector_weight_spec": SELECTOR_WEIGHT_SPEC,
            "lab_parameters": LAB_PARAMETERS,
        }
        if extra:
            data.update(extra)
        atomic_write_text(self.summary_path, json.dumps(data, ensure_ascii=True, indent=2))

    @staticmethod
    def _sha256(path: Path) -> str:
        digest = hashlib.sha256()
        with open(path, "rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def _upload_file(self, path: Path, path_in_repo: str | None = None) -> str:
        if not self.enabled or self.api is None:
            return "disabled"
        if not path.exists() or not path.is_file():
            return "missing"
        path_in_repo = path_in_repo or f"runs/{self.run_id}/{path.name}"
        payload = path.read_bytes()
        assert_no_sensitive_bytes(payload, context=f"HF upload {path_in_repo}")
        digest = hashlib.sha256(payload).hexdigest()
        signature = (len(payload), digest)
        if self._uploaded_signatures.get(path_in_repo) == signature:
            return "unchanged"
        for attempt in range(1, 5):
            try:
                with warnings.catch_warnings():
                    warnings.filterwarnings("ignore", message=r".*No files have been modified since last commit.*")
                    commit = self.api.upload_file(
                        path_or_fileobj=io.BytesIO(payload),
                        path_in_repo=path_in_repo,
                        repo_id=self.repo_id,
                        repo_type="dataset",
                        token=self.token,
                    )
                self._uploaded_signatures[path_in_repo] = signature
                self._uploaded_receipts[path_in_repo] = {
                    "size_bytes": len(payload),
                    "sha256": digest,
                    "commit_oid": getattr(commit, "oid", None),
                }
                return "uploaded"
            except Exception as exc:
                rate_limited = "429" in repr(exc) or "Too Many Requests" in repr(exc)
                if attempt < 4 and rate_limited:
                    time.sleep(min(60, 2 ** attempt * 5))
                    continue
                raise
        return "failed"

    def sync_once(
        self,
        extra_paths: list[Path] | None = None,
        heartbeat_status: str | None = "running",
        *,
        wait_for_lock: bool = False,
    ) -> bool:
        if not self.enabled:
            return False
        acquired = (
            self._sync_lock.acquire(timeout=60)
            if wait_for_lock
            else self._sync_lock.acquire(blocking=False)
        )
        if not acquired:
            return False
        errors_before = self._sync_errors
        try:
            self._sync_once_impl(extra_paths=extra_paths, heartbeat_status=heartbeat_status)
        finally:
            self._sync_lock.release()
        return self._sync_errors == errors_before

    def _sync_once_impl(self, extra_paths: list[Path] | None = None, heartbeat_status: str | None = "running") -> None:
        if not self.enabled:
            return
        if heartbeat_status is not None:
            self.write_heartbeat(heartbeat_status)
        records = []
        targets = [
            (path, f"runs/{self.run_id}/{path.name}")
            for path in [
                self.bootstrap_path, self.events_path, self.stdout_path,
                self.stderr_path, self.heartbeat_path, self.summary_path,
            ]
        ]
        seen_remote = {remote for _, remote in targets}
        for value in extra_paths or []:
            path = Path(value)
            path_identity = hashlib.sha256(str(path.resolve()).encode("utf-8")).hexdigest()[:12]
            remote = f"runs/{self.run_id}/artifacts/{path_identity}_{path.name}"
            if remote in seen_remote:
                records.append({"local_path": str(path), "remote_path": remote, "status": "duplicate_remote_skipped"})
                continue
            seen_remote.add(remote)
            targets.append((path, remote))
        for path, remote in targets:
            record = {"local_path": str(path), "remote_path": remote}
            try:
                record["status"] = self._upload_file(path, remote)
                receipt = self._uploaded_receipts.get(remote)
                if receipt:
                    record.update(receipt)
            except Exception as exc:
                self._sync_errors += 1
                record["status"] = "error"
                record["error"] = f"{type(exc).__name__}: {exc}"[:500]
                self.event("hf_sync_file_error", {**record, "count": self._sync_errors}, upload=False)
            records.append(record)
        index = {
            "schema_version": 1,
            "run_id": self.run_id,
            "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "sync_errors": self._sync_errors,
            "records": records,
        }
        atomic_write_text(self.artifact_index_path, json.dumps(index, ensure_ascii=True, indent=2))
        try:
            self._upload_file(self.artifact_index_path)
            self._upload_file(self.events_path)
        except Exception as exc:
            self._sync_errors += 1
            self.event("hf_sync_index_error", {"error": repr(exc), "count": self._sync_errors}, upload=False)

    def _loop(self) -> None:
        while not self._stop.wait(self.sync_seconds):
            try:
                self.sync_once()
            except Exception as exc:
                self._sync_errors += 1
                self.event("hf_sync_loop_error", {"error": repr(exc), "count": self._sync_errors}, upload=False)

    def stop(self, status: str = "stopped", extra_paths: list[Path] | None = None, extra: dict | None = None) -> None:
        self._stop.set()
        if not self.enabled:
            self.write_summary(status, extra=extra)
            self.event(f"hf_logging_{status}_local_only", extra or {}, upload=False)
            return
        if self._thread is not None and self._thread is not threading.current_thread():
            self._thread.join(timeout=min(30, self.sync_seconds))
        self.update_runtime_state(active_phase=status, active_command=None, command_started_epoch=None)
        self.write_summary(status, extra=extra)
        self.event(f"hf_logging_{status}", extra or {}, upload=False)
        self.write_heartbeat(status, extra=extra)
        sys.stdout.flush()
        sys.stderr.flush()
        final_sync_ok = self.sync_once(
            extra_paths=extra_paths,
            heartbeat_status=None,
            wait_for_lock=True,
        )
        if not final_sync_ok:
            self.event("hf_final_sync_retry", {"status": status}, upload=False)
            self.write_heartbeat(status, extra={**(extra or {}), "final_sync_ok": False})
            final_sync_ok = self.sync_once(
                extra_paths=extra_paths,
                heartbeat_status=None,
                wait_for_lock=True,
            )
        if not final_sync_ok:
            raise RuntimeError(f"HF final sync failed for status={status}")


class DriveLogMirror:
    def __init__(self, source_dir: Path, dest_dir: Path, sync_seconds: int):
        self.source_dir = Path(source_dir)
        self.dest_dir = Path(dest_dir)
        self.sync_seconds = max(10, int(sync_seconds))
        self.dest_dir.mkdir(parents=True, exist_ok=True)
        self._stop = threading.Event()
        self._thread = None
        self._lock = threading.Lock()
        self._copied_signatures = {}
        self._sync_errors = 0

    def _copy_file(self, path: Path) -> None:
        if not path.exists() or not path.is_file():
            return
        rel = path.relative_to(self.source_dir)
        dest = self.dest_dir / rel
        payload = path.read_bytes()
        assert_no_sensitive_bytes(payload, context=f"Drive mirror {rel.as_posix()}")
        signature = (len(payload), hashlib.sha256(payload).hexdigest())
        key = rel.as_posix()
        if self._copied_signatures.get(key) == signature:
            return
        atomic_write_bytes(dest, payload)
        self._copied_signatures[key] = signature

    def sync_once(self, *, wait_for_lock: bool = False) -> bool:
        acquired = self._lock.acquire(timeout=60) if wait_for_lock else self._lock.acquire(blocking=False)
        if not acquired:
            return False
        errors_before = self._sync_errors
        try:
            if not self.source_dir.exists():
                return False
            for path in self.source_dir.rglob("*"):
                self._copy_file(path)
        except Exception as exc:
            self._sync_errors += 1
            if HF_BRIDGE is not None:
                HF_BRIDGE.event("drive_log_sync_error", {
                    "error": repr(exc), "count": self._sync_errors, "dest_dir": str(self.dest_dir),
                })
        finally:
            self._lock.release()
        return self._sync_errors == errors_before

    def _loop(self) -> None:
        while not self._stop.wait(self.sync_seconds):
            self.sync_once()

    def start(self) -> None:
        self.sync_once()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self) -> None:
        self._stop.set()
        if self._thread is not None and self._thread is not threading.current_thread():
            self._thread.join(timeout=min(30, self.sync_seconds))
        if not self.sync_once(wait_for_lock=True):
            raise RuntimeError(f"Drive final log sync failed: {self.dest_dir}")


SENSITIVE_CHILD_ENV_NAMES = {
    "HF_TOKEN", "HF_KEY", "HUGGING_FACE_HUB_TOKEN", "OPENROUTER_API_KEY",
    "KAGGLE_USERNAME", "KAGGLE_KEY", "GH_TOKEN", "GITHUB_TOKEN",
}


CONTROLLED_CHILD_ENV_PREFIXES = (
    "ARC_QWEN_", "ARC_PROBE_", "ARC_AUTOLEARN_", "ARC_SELECTOR_",
    "ARC_MAX_", "ARC_MISSING_", "ARC_DOWNLOAD_", "ARC_UNSLOTH_",
    "ARC_KAGGLE_", "ARC_ALLOW_",
)

PACKAGE_ENV_EXACT_BLOCKLIST = {
    "PYTHONHOME", "PYTHONPATH", "PYTHONSTARTUP", "PYTHONUSERBASE",
}
TRUSTED_EXECUTABLE_PATH = ":".join((
    str(Path(sys.executable).resolve().parent),
    "/usr/local/cuda/bin", "/usr/local/bin", "/usr/bin", "/bin",
))


def safe_package_install_env(base: dict | None = None) -> dict:
    child = dict(os.environ if base is None else base)
    for name in list(child):
        if name in PACKAGE_ENV_EXACT_BLOCKLIST or name.startswith("PIP_"):
            child.pop(name, None)
    child["PYTHONNOUSERSITE"] = "1"
    child["PATH"] = TRUSTED_EXECUTABLE_PATH
    return child


def sanitized_child_env(base: dict | None = None, *, allow: set[str] | None = None) -> dict:
    child = safe_package_install_env(base)
    allowed = set(allow or ())
    for name in list(child):
        if name in SENSITIVE_CHILD_ENV_NAMES and name not in allowed:
            child.pop(name, None)
        elif name not in allowed and name.startswith(CONTROLLED_CHILD_ENV_PREFIXES):
            child.pop(name, None)
    return child


SENSITIVE_RUNTIME_VALUES: set[str] = set()


def register_sensitive_value(value: str | None) -> str | None:
    if value and len(value) >= 6:
        SENSITIVE_RUNTIME_VALUES.add(value)
    return value


def _redact_sensitive_text(text: str) -> str:
    redacted = str(text)
    for name in SENSITIVE_CHILD_ENV_NAMES:
        value = os.environ.get(name)
        if value and len(value) >= 6:
            SENSITIVE_RUNTIME_VALUES.add(value)
    for value in sorted(SENSITIVE_RUNTIME_VALUES, key=len, reverse=True):
        redacted = redacted.replace(value, "<sensitive:redacted>")
    return redacted


def assert_no_sensitive_bytes(payload: bytes, *, context: str) -> None:
    if not isinstance(payload, bytes):
        raise TypeError(f"payload must be bytes: context={context}")
    for name in SENSITIVE_CHILD_ENV_NAMES:
        value = os.environ.get(name)
        if value and len(value) >= 6:
            SENSITIVE_RUNTIME_VALUES.add(value)
    for value in SENSITIVE_RUNTIME_VALUES:
        if value.encode("utf-8") in payload:
            raise RuntimeError(
                f"sensitive value blocked before persistence: context={context}"
            )


def _descendant_pids(root_pid: int) -> list[int]:
    if os.name == "nt" or root_pid <= 1:
        return []
    children: dict[int, list[int]] = {}
    for status_path in Path("/proc").glob("[0-9]*/status"):
        try:
            pid = int(status_path.parent.name)
            ppid_line = next(
                line for line in status_path.read_text(encoding="ascii", errors="strict").splitlines()
                if line.startswith("PPid:")
            )
            parent_pid = int(ppid_line.split(":", 1)[1].strip())
        except (OSError, StopIteration, UnicodeError, ValueError):
            continue
        children.setdefault(parent_pid, []).append(pid)
    descendants: list[int] = []
    stack = list(children.get(root_pid, []))
    while stack:
        pid = stack.pop()
        if pid in descendants or pid == os.getpid():
            continue
        descendants.append(pid)
        stack.extend(children.get(pid, []))
    return descendants


def _process_group_ids(pids: list[int]) -> list[int]:
    if os.name == "nt":
        return []
    try:
        supervisor_pgid = os.getpgrp()
    except OSError:
        supervisor_pgid = -1
    groups: list[int] = []
    for pid in pids:
        try:
            pgid = os.getpgid(pid)
        except (OSError, ProcessLookupError, PermissionError):
            continue
        if pgid > 1 and pgid != supervisor_pgid and pgid not in groups:
            groups.append(pgid)
    return groups


def _signal_process_tree(
    root_pid: int,
    sig: int,
    *,
    retained_pids: list[int] | None = None,
    retained_pgids: list[int] | None = None,
) -> tuple[list[int], list[int]]:
    targets = [*_descendant_pids(root_pid), root_pid]
    if retained_pids:
        targets = list(dict.fromkeys([*targets, *retained_pids]))
    groups = _process_group_ids(targets)
    if retained_pgids:
        groups = list(dict.fromkeys([*groups, *retained_pgids]))
    if os.name != "nt":
        for pgid in groups:
            try:
                os.killpg(pgid, sig)
            except (OSError, ProcessLookupError, PermissionError):
                pass
    for pid in targets:
        try:
            os.kill(pid, sig)
        except (OSError, ProcessLookupError, PermissionError):
            pass
    return targets, groups


def run_streamed(
    cmd: list[str],
    *,
    cwd: str | None = None,
    env: dict | None = None,
    check: bool = True,
    label: str | None = None,
    timeout_seconds: int | float | None = None,
    idle_timeout_seconds: int | float | None = None,
) -> subprocess.CompletedProcess:
    label = label or Path(cmd[0]).name
    safe_cmd = [str(x) for x in cmd]
    command_started = time.monotonic()
    last_output = [command_started]
    timeout_state: dict[str, str | None] = {"reason": None}
    tail = deque(maxlen=240)
    if HF_BRIDGE is not None:
        HF_BRIDGE.update_runtime_state(
            active_phase=label,
            active_command=[_redact_sensitive_text(part) for part in safe_cmd],
            command_started_epoch=time.time(),
            last_progress_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        )
        HF_BRIDGE.event("command_start", {
            "label": label,
            "cmd": [_redact_sensitive_text(part) for part in safe_cmd],
            "cwd": cwd,
            "timeout_seconds": timeout_seconds,
            "idle_timeout_seconds": idle_timeout_seconds,
        })
    print(f"[cmd:{label}] {' '.join(_redact_sensitive_text(part) for part in safe_cmd)}")
    allowed_sensitive = (
        {"KAGGLE_USERNAME", "KAGGLE_KEY"}
        if label == "stage_kaggle_qwen_unsloth"
        else set()
    )
    child_env = sanitized_child_env(allow=allowed_sensitive)
    if env is not None:
        unexpected_sensitive = (set(env) & SENSITIVE_CHILD_ENV_NAMES) - allowed_sensitive
        if unexpected_sensitive:
            raise RuntimeError(
                f"command {label!r} requested disallowed sensitive environment names: "
                f"{sorted(unexpected_sensitive)}"
            )
        child_env.update({str(key): str(value) for key, value in env.items()})
    proc = subprocess.Popen(
        safe_cmd,
        cwd=cwd,
        env=child_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
        bufsize=1,
        start_new_session=True,
    )
    watchdog_stop = threading.Event()

    def terminate_group(reason: str) -> None:
        if timeout_state["reason"] is not None:
            return
        timeout_state["reason"] = reason
        terminated_pids: list[int] = []
        terminated_pgids: list[int] = []
        if os.name == "nt":
            proc.terminate()
        else:
            terminated_pids, terminated_pgids = _signal_process_tree(
                proc.pid, signal.SIGTERM
            )
        if proc.poll() is None:
            time.sleep(5)
        if os.name == "nt":
            if proc.poll() is None:
                proc.kill()
        else:
            # Descendants may survive TERM even when the session leader exits.
            _signal_process_tree(
                proc.pid,
                getattr(signal, "SIGKILL", 9),
                retained_pids=terminated_pids,
                retained_pgids=terminated_pgids,
            )

    def watchdog() -> None:
        while not watchdog_stop.wait(2):
            now = time.monotonic()
            if timeout_seconds is not None and now - command_started > float(timeout_seconds):
                terminate_group("total_timeout")
                return
            if idle_timeout_seconds is not None and now - last_output[0] > float(idle_timeout_seconds):
                terminate_group("idle_timeout")
                return

    watchdog_thread = threading.Thread(target=watchdog, daemon=True)
    watchdog_thread.start()
    assert proc.stdout is not None
    suppressed_noncritical = 0
    encoding_replacement_count = 0
    try:
        for raw_line in proc.stdout:
            last_output[0] = time.monotonic()
            encoding_replacement_count += raw_line.count("\ufffd")
            line = _redact_sensitive_text(raw_line)
            tail.append(line)
            if HF_BRIDGE is not None:
                HF_BRIDGE.update_runtime_state(
                    last_progress_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
                )
            if is_noncritical_log_line(line):
                suppressed_noncritical += 1
                continue
            print(line, end="")
        rc = proc.wait()
    except BaseException:
        terminate_group("stream_exception")
        try:
            proc.stdout.close()
        finally:
            try:
                proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait(timeout=15)
        raise
    finally:
        watchdog_stop.set()
        watchdog_thread.join(timeout=10)
    if timeout_state["reason"] is not None:
        rc = 124
    if encoding_replacement_count:
        print(f"[cmd:{label}] UTF-8 decoding replacements={encoding_replacement_count}")
        if rc == 0:
            rc = 86
    if suppressed_noncritical:
        print(f"[cmd:{label}] suppressed_noncritical_lines={suppressed_noncritical}")
        if HF_BRIDGE is not None:
            HF_BRIDGE.event("command_suppressed_noncritical_lines", {"label": label, "count": suppressed_noncritical})
    output_tail = "".join(tail)
    print(f"[cmd:{label}] exit={rc} stop_reason={timeout_state['reason'] or 'process_exit'}")
    if HF_BRIDGE is not None:
        command_elapsed_s = round(time.monotonic() - command_started, 3)
        HF_BRIDGE.update_runtime_state(
            active_phase="idle" if rc == 0 else "failed",
            active_command=None,
            command_started_epoch=None,
            last_command=label,
            last_command_returncode=rc,
            last_command_elapsed_s=command_elapsed_s,
        )
        HF_BRIDGE.event("command_end", {
            "label": label,
            "returncode": rc,
            "duration_s": command_elapsed_s,
            "stop_reason": timeout_state["reason"] or "process_exit",
            "output_tail": output_tail[-12000:],
        }, upload=(rc != 0))
    if check and rc != 0:
        raise RuntimeError(
            f"command {label!r} failed rc={rc} stop_reason={timeout_state['reason'] or 'process_exit'}\n"
            f"output_tail:\n{output_tail[-12000:]}"
        )
    return subprocess.CompletedProcess(safe_cmd, rc, stdout=output_tail)


EARLY_FAILURE_PATH = Path("/content/arc_p147_early_failure.json")


def _record_early_failure(exc_type, exc, tb) -> str:
    traceback_text = _redact_sensitive_text(
        "".join(traceback.format_exception(exc_type, exc, tb))[-20000:]
    )
    payload = {
        "phase": "before_full_remote_logging",
        "error_type": getattr(exc_type, "__name__", str(exc_type)),
        "error": _redact_sensitive_text(repr(exc)),
        "traceback": traceback_text,
        "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    try:
        atomic_write_text(
            EARLY_FAILURE_PATH,
            json.dumps(payload, ensure_ascii=True, indent=2, sort_keys=True),
        )
    except Exception as persistence_exc:
        print(
            _redact_sensitive_text(
                f"early failure persistence error: {type(persistence_exc).__name__}: {persistence_exc}"
            ),
            file=sys.__stderr__,
            flush=True,
        )
    print("P147 EARLY FAILURE", traceback_text, file=sys.__stderr__, flush=True)
    return traceback_text


def _early_colab_excepthook(exc_type, exc, tb):
    _record_early_failure(exc_type, exc, tb)
    sys.__excepthook__(exc_type, exc, tb)


def _early_ipython_failure_handler(self, etype, value, tb, tb_offset=None):
    _record_early_failure(etype, value, tb)
    return traceback.format_exception(etype, value, tb)


sys.excepthook = _early_colab_excepthook
try:
    get_ipython().set_custom_exc((BaseException,), _early_ipython_failure_handler)
except Exception as exc:
    raise RuntimeError(f"Could not install early IPython failure handler: {exc}") from exc


section("1. Runtime probe")
try:
    import torch
except Exception as exc:
    raise RuntimeError("PyTorch is not importable in this Colab runtime") from exc

print("python", sys.version)
P147_EXPECTED_PYTHON = (3, 12)
if sys.version_info[:2] != P147_EXPECTED_PYTHON:
    raise RuntimeError(
        f"P147 requires Python {P147_EXPECTED_PYTHON[0]}.{P147_EXPECTED_PYTHON[1]} "
        f"for the validated dependency contract; got {sys.version.split()[0]}"
    )
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("cuda available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime -> Change runtime type -> GPU")

gpu_name = torch.cuda.get_device_name(0)
print("gpu", gpu_name)
nvidia_smi = subprocess.run(
    ["/usr/bin/nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
    capture_output=True,
    check=False,
    timeout=15,
)
if nvidia_smi.returncode != 0:
    raise RuntimeError(f"nvidia-smi failed rc={nvidia_smi.returncode}: {nvidia_smi.stderr[-2000:]}")
print(nvidia_smi.stdout.strip())

gpu_count = torch.cuda.device_count()
gpu_capability = torch.cuda.get_device_capability(0)
gpu_bf16_supported = bool(torch.cuda.is_bf16_supported())
if int(FORCE_GPU_COUNT) > gpu_count:
    raise RuntimeError(f"FORCE_GPU_COUNT={FORCE_GPU_COUNT} exceeds visible CUDA devices={gpu_count}")
if TRAIN_PRECISION == "bf16" and not gpu_bf16_supported:
    raise RuntimeError("TRAIN_PRECISION=bf16 is unsupported by this GPU")
RESOLVED_TRAIN_PRECISION = (
    "bf16" if TRAIN_PRECISION == "auto" and gpu_bf16_supported
    else "fp16" if TRAIN_PRECISION == "auto"
    else TRAIN_PRECISION
)
print("precision preflight", json.dumps({
    "gpu_count": gpu_count, "capability": gpu_capability,
    "bf16_supported": gpu_bf16_supported, "resolved": RESOLVED_TRAIN_PRECISION,
}, sort_keys=True))
if "L4" not in gpu_name:
    print("[runtime-note] GPU is not L4; use result as functional evidence, not Kaggle timing proof.")
    if REQUIRE_L4_TIMING:
        raise RuntimeError("REQUIRE_L4_TIMING is enabled, but the selected GPU is not L4")


def secret(name: str) -> str | None:
    try:
        value = userdata.get(name)
        return register_sensitive_value(value) if value else None
    except Exception as exc:
        detail = _redact_sensitive_text(f"{type(exc).__name__}: {exc}")
        print(f"[secret:{name}] unavailable: {detail}")
        return None


section("2. Mount Drive and unpack bundle")
from google.colab import drive, userdata
from importlib import metadata as _bootstrap_metadata

_HF_BRIDGE_VERSION = "0.36.2"
subprocess.run(
    [sys.executable, "-m", "pip", "--isolated", "install", "-q", "--disable-pip-version-check", f"huggingface_hub=={_HF_BRIDGE_VERSION}"],
    env=safe_package_install_env(),
    check=True,
    timeout=300,
)
if _bootstrap_metadata.version("huggingface-hub") != _HF_BRIDGE_VERSION:
    raise RuntimeError("HF bridge bootstrap version mismatch")

# Start HF logging before Drive so a DriveFS failure is observable and nonfatal.
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN") or secret("HF_TOKEN") or secret("HF_KEY") or ""
register_sensitive_value(os.environ.get("HF_TOKEN"))
HF_BRIDGE = HFLogBridge(
    token=os.environ.get("HF_TOKEN", ""),
    run_id=RUN_ID,
    dataset_repo=HF_LOG_DATASET,
    sync_seconds=HF_LOG_SYNC_SECONDS,
)
DRIVE_LOG_MIRROR = None


def _colab_excepthook(exc_type, exc, tb):
    finalizer_errors = []
    if HF_BRIDGE is not None:
        try:
            HF_BRIDGE.event("run_failed", {
                "error": _redact_sensitive_text(repr(exc)),
                "traceback": _redact_sensitive_text("".join(traceback.format_exception(exc_type, exc, tb))[-12000:]),
            }, upload=HF_BRIDGE.enabled)
        except Exception as hook_exc:
            finalizer_errors.append(_redact_sensitive_text(f"hf_event:{type(hook_exc).__name__}:{hook_exc}"))
        try:
            HF_BRIDGE.stop("failed", extra={"error": _redact_sensitive_text(repr(exc))})
        except Exception as hook_exc:
            finalizer_errors.append(_redact_sensitive_text(f"hf_stop:{type(hook_exc).__name__}:{hook_exc}"))
    try:
        (Path.home() / ".kaggle" / "kaggle.json").unlink(missing_ok=True)
        for credential_name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            os.environ.pop(credential_name, None)
    except Exception as hook_exc:
        finalizer_errors.append(_redact_sensitive_text(f"credential_cleanup:{type(hook_exc).__name__}:{hook_exc}"))
    if DRIVE_LOG_MIRROR is not None:
        try:
            DRIVE_LOG_MIRROR.stop()
        except Exception as hook_exc:
            finalizer_errors.append(_redact_sensitive_text(f"drive_stop:{type(hook_exc).__name__}:{hook_exc}"))
    if finalizer_errors:
        print("failure finalizer errors", json.dumps(finalizer_errors), file=sys.__stderr__)
    sys.__excepthook__(exc_type, exc, tb)


sys.excepthook = _colab_excepthook

def _ipython_failure_handler(self, etype, value, tb, tb_offset=None):
    try:
        _colab_excepthook(etype, value, tb)
    except Exception as hook_exc:
        print(_redact_sensitive_text(f"failure finalizer error: {type(hook_exc).__name__}: {hook_exc}"), file=sys.__stderr__)
    return traceback.format_exception(etype, value, tb)

try:
    get_ipython().set_custom_exc((BaseException,), _ipython_failure_handler)
except Exception as exc:
    raise RuntimeError(f"Could not install IPython failure finalizer: {exc}") from exc

sys.stdout = TeeStream(sys.__stdout__, HF_BRIDGE.stdout_path)
sys.stderr = TeeStream(sys.__stderr__, HF_BRIDGE.stderr_path)

DRIVE_AVAILABLE = False
DRIVE_MOUNT_ERROR = None


def probe_drive_writable():
    root = Path("/content/drive/MyDrive")
    if not root.is_dir():
        return False, "MyDrive directory is absent"
    probe = root / f".arc2016_drive_probe_{os.getpid()}"
    try:
        probe.write_text("ok", encoding="ascii")
        if probe.read_text(encoding="ascii") != "ok":
            return False, "Drive write probe content mismatch"
        probe.unlink()
        return True, None
    except Exception as exc:
        try:
            probe.unlink(missing_ok=True)
        except Exception:
            pass
        return False, f"{type(exc).__name__}: {exc}"


DRIVE_AVAILABLE, DRIVE_MOUNT_ERROR = probe_drive_writable()
if DRIVE_AVAILABLE:
    print("Drive already mounted and accessible")
elif TRY_DRIVE_MOUNT:
    try:
        drive.mount("/content/drive", timeout_ms=180000)
        DRIVE_AVAILABLE, DRIVE_MOUNT_ERROR = probe_drive_writable()
        if not DRIVE_AVAILABLE:
            print("[runtime-note] Drive mounted but failed writable probe; using HF + /content")
    except Exception as exc:
        DRIVE_MOUNT_ERROR = f"{type(exc).__name__}: {exc}"
        print("[runtime-note] Drive mount unavailable; continuing with embedded bundle and HF logs")
        print("drive mount detail", DRIVE_MOUNT_ERROR)
else:
    DRIVE_MOUNT_ERROR = "disabled_by_TRY_DRIVE_MOUNT"
    print("[runtime-note] Drive mount skipped; using embedded bundle and HF logs")

if DRIVE_AVAILABLE:
    drive_my_root = Path("/content/drive/MyDrive").resolve(strict=True)
    requested_drive_root = Path(DRIVE_LOG_ROOT).resolve(strict=False)
    if requested_drive_root == drive_my_root or drive_my_root not in requested_drive_root.parents:
        raise ValueError(
            f"DRIVE_LOG_ROOT must be a subdirectory of MyDrive: {requested_drive_root}"
        )
    drive_log_dest = requested_drive_root / RUN_ID
    if drive_log_dest.exists():
        raise FileExistsError(f"Drive run log destination already exists: {drive_log_dest}")
    DRIVE_LOG_MIRROR = DriveLogMirror(
        HF_BRIDGE.log_dir,
        drive_log_dest,
        DRIVE_LOG_SYNC_SECONDS,
    )
    DRIVE_LOG_MIRROR.start()
    print("drive live log dir", DRIVE_LOG_MIRROR.dest_dir)
else:
    DRIVE_LOG_MIRROR = None
    print("Drive mirror disabled for this run; HF is the remote evidence store")
if HF_BRIDGE.enabled:
    print("hf log repo", HF_BRIDGE.repo_id)
    print("hf log url", HF_BRIDGE.repo_url)
else:
    print("hf remote logging disabled or unavailable; logs remain local to this runtime")
if not DRIVE_AVAILABLE and not HF_BRIDGE.enabled:
    raise RuntimeError(
        "Drive is unavailable and HF logging could not start. Check HF_TOKEN/HF_KEY "
        "before running a long experiment without a remote evidence store."
    )
print("lab parameters pre-episode", json.dumps(LAB_PARAMETERS, sort_keys=True))
HF_BRIDGE.event(
    "lab_parameters_pre_episode",
    {"phase": "pre_episode_identity", "parameters": dict(LAB_PARAMETERS)},
    upload=HF_BRIDGE.enabled,
)
HF_BRIDGE.event("drive_mount_status", {
    "available": DRIVE_AVAILABLE,
    "error": DRIVE_MOUNT_ERROR,
    "try_drive_mount": bool(TRY_DRIVE_MOUNT),
}, upload=HF_BRIDGE.enabled)
if DRIVE_LOG_MIRROR is not None:
    HF_BRIDGE.event("drive_log_mirror_started", {
        "dest_dir": str(DRIVE_LOG_MIRROR.dest_dir),
        "sync_seconds": DRIVE_LOG_MIRROR.sync_seconds,
    }, upload=HF_BRIDGE.enabled)


local_bundle = Path("/content") / BUNDLE_NAME
try:
    embedded_payload = base64.b64decode(EMBEDDED_BUNDLE_B64, validate=True)
except Exception as exc:
    raise RuntimeError(f"Embedded bundle base64 is invalid: {type(exc).__name__}: {exc}") from exc
embedded_sha256 = hashlib.sha256(embedded_payload).hexdigest()
if embedded_sha256 != EMBEDDED_BUNDLE_SHA256:
    raise RuntimeError(
        "Embedded bundle SHA-256 mismatch: "
        f"expected={EMBEDDED_BUNDLE_SHA256} actual={embedded_sha256}"
    )
atomic_write_bytes(local_bundle, embedded_payload)
bundle_path = local_bundle
print("bundle source embedded")
print("bundle sha256", embedded_sha256)
print("bundle local path", local_bundle)
print("bundle local size", local_bundle.stat().st_size)

if Path(ROOT_DIR).exists():
    shutil.rmtree(ROOT_DIR)
Path(ROOT_DIR).mkdir(parents=True, exist_ok=True)


def safe_extract_bundle(bundle_zip: zipfile.ZipFile, root: Path) -> None:
    """Extract the sealed bundle while rejecting ambiguous or hostile members."""
    members = bundle_zip.infolist()
    if len(members) > 5000:
        raise RuntimeError(f"Bundle member count exceeds limit: {len(members)}")
    total_uncompressed = sum(int(member.file_size) for member in members)
    if total_uncompressed > 2 * 1024 * 1024 * 1024:
        raise RuntimeError(f"Bundle uncompressed size exceeds limit: {total_uncompressed}")
    root = root.resolve(strict=True)
    seen: set[str] = set()
    for member in members:
        raw_name = member.filename
        if not raw_name or "\x00" in raw_name or raw_name.startswith(("/", "\\")):
            raise RuntimeError(f"Unsafe bundle member path: {raw_name!r}")
        normalized = raw_name.replace("\\", "/")
        parts = [part for part in normalized.split("/") if part not in {"", "."}]
        if (
            not parts
            or any(part == ".." or ":" in part for part in parts)
            or ((member.external_attr >> 16) & 0o170000) == 0o120000
        ):
            raise RuntimeError(f"Unsafe bundle member path: {raw_name!r}")
        normalized_key = "/".join(parts)
        if normalized_key in seen:
            raise RuntimeError(f"Duplicate bundle member path: {normalized_key!r}")
        seen.add(normalized_key)
        target = root.joinpath(*parts)
        if not target.resolve(strict=False).is_relative_to(root):
            raise RuntimeError(f"Bundle member escaped extraction root: {raw_name!r}")
        if member.is_dir() or raw_name.endswith(("/", "\\")):
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with bundle_zip.open(member) as src, open(target, "xb") as dst:
            shutil.copyfileobj(src, dst)


try:
    zip_names = []
    with zipfile.ZipFile(local_bundle) as bundle_zip:
        zip_names = bundle_zip.namelist()
        bad_member = bundle_zip.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Corrupt member in bundle zip: {bad_member}")
        print("bundle entries", len(zip_names))
        print("bundle first entries", zip_names[:20])
        print("bundle backslash entries", sum("\\" in name for name in zip_names))
        safe_extract_bundle(bundle_zip, Path(ROOT_DIR))
except zipfile.BadZipFile as exc:
    raise RuntimeError(
        f"Bundle is not a valid zip after Drive copy: {local_bundle} "
        f"({local_bundle.stat().st_size if local_bundle.exists() else 'missing'} bytes)"
    ) from exc
finally:
    embedded_payload = b""
    local_bundle.unlink(missing_ok=True)
    if local_bundle.exists():
        raise RuntimeError(f"extracted label-bearing bundle could not be removed: {local_bundle}")
    bundle_path = Path("/content/<deleted-label-bearing-p147-bundle>")

def extracted_tree_sample(root: Path, limit: int = 120) -> list[str]:
    if not root.exists():
        return [f"{root} does not exist"]
    rows = []
    for path in root.rglob("*"):
        try:
            rel = path.relative_to(root).as_posix()
        except ValueError:
            rel = str(path)
        rows.append(rel + ("/" if path.is_dir() else ""))
        if len(rows) >= limit:
            break
    return rows


def resolve_arc2_root(root: Path) -> Path:
    candidates = []
    direct = root / "arc2"
    if direct.exists():
        candidates.append(direct)
    for marker in root.rglob("qwen_worker_throughput.py"):
        if marker.parent.name == "hf":
            candidates.append(marker.parent.parent)
    for candidate in candidates:
        if (
            (candidate / "hf" / "qwen_worker_throughput.py").exists()
            and (candidate / "kaggle_qwen_l4x4" / "qwen_ttt_worker.py").exists()
            and (candidate / "data" / "arc-agi_evaluation_challenges.json").exists()
        ):
            return candidate
    raise RuntimeError(
        "Bundle extracted, but arc2 root was not found. "
        f"root={root} zip_first_entries={zip_names[:40]} "
        f"extracted_tree_sample={extracted_tree_sample(root)}"
    )


ARC2 = resolve_arc2_root(Path(ROOT_DIR))
print("bundle", bundle_path)
print("arc2", ARC2)
print("extracted tree sample", extracted_tree_sample(Path(ROOT_DIR), limit=40))


def verify_bundle_contract(arc2: Path) -> None:
    manifest_path = arc2 / "BUNDLE_MANIFEST.json"
    if not manifest_path.is_file():
        raise RuntimeError("Bundle manifest is missing")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8", errors="strict"))
    if not isinstance(manifest, dict):
        raise RuntimeError("Bundle manifest root must be an object")
    if manifest.get("schema") != "arc-agi-2-colab-bundle-manifest-v1":
        raise RuntimeError(f"unexpected bundle manifest schema: {manifest.get('schema')!r}")
    if manifest.get("release") != "p147-20260801":
        raise RuntimeError(f"unexpected bundle release: {manifest.get('release')!r}")
    files = manifest.get("files")
    if not isinstance(files, dict) or not files:
        raise RuntimeError("Bundle manifest has no files")
    if manifest.get("file_count") != len(files):
        raise RuntimeError(
            f"bundle manifest count mismatch: declared={manifest.get('file_count')} observed={len(files)}"
        )
    actual_members = {
        "arc2/" + path.relative_to(arc2).as_posix()
        for path in arc2.rglob("*")
        if path.is_file() and path != manifest_path
    }
    if actual_members != set(files):
        missing = sorted(set(files) - actual_members)
        extra = sorted(actual_members - set(files))
        raise RuntimeError(f"bundle member inventory mismatch: missing={missing} extra={extra}")
    failures = []
    compiled = 0
    for member, expected in sorted(files.items()):
        if not str(member).startswith("arc2/"):
            failures.append(f"{member}:outside_arc2")
            continue
        if not isinstance(expected, dict) or set(expected) != {"bytes", "sha256"}:
            failures.append(f"{member}:invalid_manifest_record")
            continue
        expected_bytes = expected.get("bytes")
        expected_sha256 = expected.get("sha256")
        if isinstance(expected_bytes, bool) or not isinstance(expected_bytes, int) or expected_bytes < 0:
            failures.append(f"{member}:invalid_bytes")
            continue
        if not isinstance(expected_sha256, str) or re.fullmatch(r"[0-9a-f]{64}", expected_sha256) is None:
            failures.append(f"{member}:invalid_sha256")
            continue
        path = Path(ROOT_DIR) / Path(*PurePosixPath(member).parts)
        if not path.is_file():
            failures.append(f"{member}:missing")
            continue
        raw = path.read_bytes()
        if len(raw) != expected_bytes:
            failures.append(f"{member}:size")
        if hashlib.sha256(raw).hexdigest() != expected_sha256:
            failures.append(f"{member}:sha256")
        if path.suffix.lower() in {".py", ".json", ".jsonl", ".md", ".txt", ".yaml", ".yml"}:
            try:
                decoded = raw.decode("utf-8", errors="strict")
                if "\ufffd" in decoded:
                    failures.append(f"{member}:replacement_character")
                if path.suffix.lower() == ".py":
                    compile(decoded, str(path), "exec")
                    compiled += 1
            except (UnicodeError, SyntaxError) as exc:
                failures.append(f"{member}:{type(exc).__name__}:{exc}")
    if failures:
        raise RuntimeError(f"Bundle contract failed: {failures[:40]}")
    print("bundle manifest verified files", len(files), "compiled_python", compiled)
    return manifest


BUNDLE_MANIFEST = verify_bundle_contract(ARC2)
LABEL_VAULT_PAYLOADS: dict[str, bytes] = {}
for solution_path in sorted((ARC2 / "data").glob("*solutions*.json")):
    if solution_path.is_symlink() or not solution_path.is_file():
        raise RuntimeError(f"label source is not a regular file: {solution_path}")
    relative = solution_path.relative_to(ARC2).as_posix()
    member = f"arc2/{relative}"
    expected = BUNDLE_MANIFEST["files"].get(member)
    payload = solution_path.read_bytes()
    observed = {"bytes": len(payload), "sha256": hashlib.sha256(payload).hexdigest()}
    if expected != observed:
        raise RuntimeError(
            f"label source identity mismatch before vaulting: member={member} "
            f"expected={expected} observed={observed}"
        )
    LABEL_VAULT_PAYLOADS[member] = payload
    solution_path.unlink()
if not LABEL_VAULT_PAYLOADS:
    raise RuntimeError("no solution sources were vaulted from the sealed bundle")
remaining_vault_sources = sorted((ARC2 / "data").glob("*solutions*.json"))
if remaining_vault_sources:
    raise RuntimeError(f"solution files remain visible after vaulting: {remaining_vault_sources}")
SOURCE_TRAINING_SOLUTIONS_PAYLOAD = LABEL_VAULT_PAYLOADS[
    "arc2/data/arc-agi_training_solutions.json"
]
bootstrap_event("label_sources_vaulted", {
    "members": sorted(LABEL_VAULT_PAYLOADS),
    "file_count": len(LABEL_VAULT_PAYLOADS),
})

sys.path.insert(0, str(ARC2 / "kaggle_qwen_l4x4"))
sys.path.insert(0, str(ARC2 / "pipeline"))
for module_name in ("qwen_ttt_worker", "candidate_selector", "pre_submit_check"):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
import qwen_ttt_worker as _qwen_ttt_worker_module
import candidate_selector as _candidate_selector_module
import pre_submit_check as _pre_submit_check_module
from qwen_ttt_worker import parse_run_matrix
from candidate_selector import load_selector_weights, normalize_selector_weights
for module, member in (
    (_qwen_ttt_worker_module, "arc2/kaggle_qwen_l4x4/qwen_ttt_worker.py"),
    (_candidate_selector_module, "arc2/pipeline/candidate_selector.py"),
    (_pre_submit_check_module, "arc2/pipeline/pre_submit_check.py"),
):
    module_path = Path(module.__file__).resolve(strict=True)
    expected_path = (Path(ROOT_DIR) / Path(*PurePosixPath(member).parts)).resolve(strict=True)
    expected_sha256 = BUNDLE_MANIFEST["files"][member]["sha256"]
    if module_path != expected_path:
        raise RuntimeError(f"module provenance mismatch: module={module.__name__} path={module_path} expected={expected_path}")
    if hashlib.sha256(module_path.read_bytes()).hexdigest() != expected_sha256:
        raise RuntimeError(f"module hash mismatch after import: {module.__name__}")

if RUN_MATRIX:
    RUN_MATRIX = parse_run_matrix(json.dumps(RUN_MATRIX, separators=(",", ":"), sort_keys=True))
    QWEN_OPTIONAL_OVERRIDES["ARC_QWEN_RUN_MATRIX_JSON"] = json.dumps(RUN_MATRIX, separators=(",", ":"), sort_keys=True)
EXPECTED_QWEN_RUN_TAGS = sorted(str(run["tag"]) for run in RUN_MATRIX)
if RUN_MATRIX and (
    len(EXPECTED_QWEN_RUN_TAGS) != len(RUN_MATRIX)
    or len(set(EXPECTED_QWEN_RUN_TAGS)) != len(EXPECTED_QWEN_RUN_TAGS)
):
    raise RuntimeError("run matrix tags are absent or duplicated")
LAB_PARAMETERS["expected_qwen_run_tags"] = EXPECTED_QWEN_RUN_TAGS
selector_contract = normalize_selector_weights(load_selector_weights(SELECTOR_WEIGHT_SPEC))
for selector_mode in csv_items(SELECTOR_SWEEP_MODES):
    normalize_selector_weights({"selection_mode": selector_mode})
print("early run-matrix/selector contract ok", json.dumps({
    "portfolio_runs": len(RUN_MATRIX),
    "selector_mode": selector_contract["selection_mode"],
    "sweep_modes": csv_items(SELECTOR_SWEEP_MODES),
}, sort_keys=True))


section("3. Kaggle/HF credentials")
for key in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    os.environ[key] = os.environ.get(key) or secret(key) or ""

drive_kaggle_json = Path("/content/drive/MyDrive/kaggle.json")
if (not os.environ.get("KAGGLE_USERNAME") or not os.environ.get("KAGGLE_KEY")) and drive_kaggle_json.exists():
    cfg = json.loads(drive_kaggle_json.read_text(encoding="utf-8"))
    os.environ["KAGGLE_USERNAME"] = cfg.get("username", "")
    os.environ["KAGGLE_KEY"] = cfg.get("key", "")

if not os.environ.get("KAGGLE_USERNAME") or not os.environ.get("KAGGLE_KEY"):
    raise RuntimeError("Missing Kaggle credentials. Add Colab Secrets or MyDrive/kaggle.json.")
for credential_name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    register_sensitive_value(os.environ.get(credential_name))

kaggle_dir = Path.home() / ".kaggle"
atomic_write_private_json(kaggle_dir / "kaggle.json", {
    "username": os.environ["KAGGLE_USERNAME"],
    "key": os.environ["KAGGLE_KEY"],
})
print("kaggle user", os.environ["KAGGLE_USERNAME"])
print("hf token present", bool(os.environ.get("HF_TOKEN")))

if HF_BRIDGE is not None:
    HF_BRIDGE.event("colab_runtime_ready", {
        "lab_config_version": LAB_CONFIG_VERSION,
        "experiment_id": EXPERIMENT_ID,
        "profiles": PROFILES,
        "selector_weight_spec": SELECTOR_WEIGHT_SPEC,
        "gpu": gpu_name,
        "python": sys.version,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "kaggle_user": os.environ["KAGGLE_USERNAME"],
    }, upload=HF_BRIDGE.enabled)


section("4. Atomic dependency install and ABI contract")
from importlib import metadata as importlib_metadata


def pip_check_snapshot():
    completed = subprocess.run(
        [sys.executable, "-m", "pip", "--isolated", "check"],
        text=True,
        capture_output=True,
        env=safe_package_install_env(),
        check=False,
        timeout=120,
    )
    lines = sorted({line.strip() for line in (completed.stdout + "\n" + completed.stderr).splitlines() if line.strip()})
    return {"returncode": completed.returncode, "lines": lines}


def installed_versions(names):
    result = {}
    for name in names:
        try:
            result[name] = importlib_metadata.version(name)
        except importlib_metadata.PackageNotFoundError:
            result[name] = None
    return result


pip_check_pristine = pip_check_snapshot()
tracked_packages = [
    *P147_EXPECTED_EXACT,
    *P147_EXPECTED_RUNTIME_PUBLIC,
    "gradio",
    "gradio-client",
    "hf-gradio",
]
versions_pristine = installed_versions(tracked_packages)
P147_UNUSED_CONFLICT_PACKAGES = ["gradio", "gradio-client", "hf-gradio"]
# Query the removal set directly. It may intentionally contain packages
# outside tracked_packages in future Colab images.
unused_conflicts_before = installed_versions(P147_UNUSED_CONFLICT_PACKAGES)
unused_conflicts_removed = [
    name for name, version in unused_conflicts_before.items() if version is not None
]
if unused_conflicts_removed:
    run_streamed(
        [sys.executable, "-m", "pip", "--isolated", "uninstall", "-q", "-y", *unused_conflicts_removed],
        check=True,
        label="pip_remove_unused_conflict_stack",
        timeout_seconds=600,
        idle_timeout_seconds=300,
    )
unused_conflicts_after = installed_versions(P147_UNUSED_CONFLICT_PACKAGES)
pip_check_post_removal = pip_check_snapshot()
versions_post_removal = installed_versions(tracked_packages)
removal_induced_conflicts = sorted(
    set(pip_check_post_removal["lines"]) - set(pip_check_pristine["lines"])
)
if any(unused_conflicts_after.values()) or removal_induced_conflicts:
    raise RuntimeError(
        "P147 conflict-stack removal was not clean: "
        f"remaining={unused_conflicts_after} introduced={removal_induced_conflicts}"
    )

torch_before = torch.__version__
cuda_before = torch.version.cuda
if torch_before.split("+", 1)[0] != P147_EXPECTED_RUNTIME_PUBLIC["torch"]:
    raise RuntimeError(
        f"P147 requires Colab torch public version {P147_EXPECTED_RUNTIME_PUBLIC['torch']}; "
        f"found {torch_before}. Refuse to replace the CUDA runtime in-place."
    )
pip_check_before = pip_check_post_removal
versions_before = versions_post_removal
run_streamed(
    [
        sys.executable,
        "-m",
        "pip",
        "--isolated",
        "install",
        "-q",
        "--disable-pip-version-check",
        "--progress-bar",
        "off",
        *P147_ATOMIC_SPECS,
    ],
    check=True,
    label="pip_p147_atomic_dependency_lock",
    timeout_seconds=1800,
    idle_timeout_seconds=600,
)
pip_check_after = pip_check_snapshot()
versions_after = installed_versions([*P147_EXPECTED_EXACT, *P147_EXPECTED_RUNTIME_PUBLIC])
torch_after = importlib_metadata.version("torch")
install_induced_conflicts = sorted(set(pip_check_after["lines"]) - set(pip_check_post_removal["lines"]))
new_pip_conflicts = sorted(set(pip_check_after["lines"]) - set(pip_check_pristine["lines"]))
version_errors = [
    f"{name}: expected {expected}, found {versions_after.get(name)}"
    for name, expected in P147_EXPECTED_EXACT.items()
    if versions_after.get(name) != expected
]
version_errors.extend(
    f"{name}: expected public version {expected}, found {versions_after.get(name)}"
    for name, expected in P147_EXPECTED_RUNTIME_PUBLIC.items()
    if (versions_after.get(name) or "").split("+", 1)[0] != expected
)
if torch.version.cuda != cuda_before:
    version_errors.append(f"CUDA changed unexpectedly: before={cuda_before} after={torch.version.cuda}")
pip_check_execution_errors = [
    f"{name}: returncode={snapshot.get('returncode')!r}"
    for name, snapshot in (
        ("pristine", pip_check_pristine),
        ("post_removal", pip_check_post_removal),
        ("after", pip_check_after),
    )
    if snapshot.get("returncode") not in {0, 1}
]
version_errors.extend(pip_check_execution_errors)

smoke_code = r"""
import os
import torch
import unsloth
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments
import transformers, peft, datasets, accelerate, trl, bitsandbytes, xformers, sklearn, tokenizers, torchvision, triton, torchao
from transformers import Qwen3Config, Qwen3ForCausalLM
from peft import LoraConfig, get_peft_model
config = Qwen3Config(
    vocab_size=128, hidden_size=32, intermediate_size=64,
    num_hidden_layers=1, num_attention_heads=4, num_key_value_heads=2,
    head_dim=8, max_position_embeddings=128,
)
model = Qwen3ForCausalLM(config)
model = get_peft_model(model, LoraConfig(r=2, lora_alpha=4, target_modules=["q_proj"]))
assert any(parameter.requires_grad for parameter in model.parameters())
model = model.cuda()
for parameter in model.parameters():
    if parameter.dtype == torch.float32 and not parameter.requires_grad:
        parameter.data = parameter.data.half()
assert all(parameter.dtype == torch.float32 for parameter in model.parameters() if parameter.requires_grad)
from accelerate import Accelerator
optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=1e-4)
accelerator = Accelerator(mixed_precision=os.environ["ARC_SMOKE_PRECISION"])
model, optimizer = accelerator.prepare(model, optimizer)
tokens = torch.tensor([[1, 2, 3, 4]], device=accelerator.device)
with accelerator.autocast():
    loss = model(input_ids=tokens, labels=tokens).loss
accelerator.backward(loss)
optimizer.step()
optimizer.zero_grad(set_to_none=True)
print("P147_QWEN3_PEFT_LORA_SMOKE_OK")
print("P147_FP16_OPTIMIZER_STEP_SMOKE_OK")
"""
try:
    smoke_env = sanitized_child_env()
    smoke_env["ARC_SMOKE_PRECISION"] = RESOLVED_TRAIN_PRECISION
    import_smoke = subprocess.run(
        [sys.executable, "-c", smoke_code],
        env=smoke_env,
        text=True,
        capture_output=True,
        check=False,
        timeout=180,
    )
except subprocess.TimeoutExpired as exc:
    import_smoke = subprocess.CompletedProcess(
        exc.cmd,
        124,
        stdout=str(exc.stdout or ""),
        stderr=f"dependency smoke timed out after {exc.timeout}s; partial_stderr={exc.stderr or ''}",
    )
dependency_contract = {
    "schema": "arc-agi-2-colab-dependency-contract-v1",
    "lab_config_version": LAB_CONFIG_VERSION,
    "atomic_specs": P147_ATOMIC_SPECS,
    "unused_conflicts_before": unused_conflicts_before,
    "unused_conflicts_removed": unused_conflicts_removed,
    "unused_conflicts_after": unused_conflicts_after,
    "pip_check_pristine": pip_check_pristine,
    "pip_check_post_removal": pip_check_post_removal,
    "versions_pristine": versions_pristine,
    "versions_post_removal": versions_post_removal,
    "removal_induced_conflicts": removal_induced_conflicts,
    "install_induced_conflicts": install_induced_conflicts,
    "torch_before": torch_before,
    "torch_after": torch_after,
    "cuda_before": cuda_before,
    "cuda_after": torch.version.cuda,
    "versions_before": versions_before,
    "versions_after": versions_after,
    "pip_check_before": pip_check_before,
    "pip_check_after": pip_check_after,
    "new_pip_conflicts": new_pip_conflicts,
    "pip_check_execution_errors": pip_check_execution_errors,
    "version_errors": version_errors,
    "import_smoke": {
        "returncode": import_smoke.returncode,
        "stdout": import_smoke.stdout[-4000:],
        "stderr": import_smoke.stderr[-8000:],
    },
}
dependency_contract["torchao_required_by_unsloth_zoo"] = True
dependency_contract["torchao_compatibility"] = "torchao-0.17.0 supports torch-2.11+"
dependency_contract["loaded_huggingface_hub_version"] = getattr(sys.modules.get("huggingface_hub"), "__version__", None)
if dependency_contract["loaded_huggingface_hub_version"] != P147_EXPECTED_EXACT["huggingface-hub"]:
    version_errors.append("loaded huggingface_hub module version differs from locked distribution")
dependency_contract["version_errors"] = list(version_errors)
atomic_write_text(DEPENDENCY_CONTRACT_PATH, json.dumps(dependency_contract, indent=2, sort_keys=True))
print("dependency contract", json.dumps(dependency_contract, sort_keys=True))
if HF_BRIDGE is not None:
    HF_BRIDGE.event("p147_dependency_contract", dependency_contract, upload=True)
if pip_check_after["returncode"] != 0 or new_pip_conflicts or version_errors or import_smoke.returncode != 0:
    raise RuntimeError(
        "P147 dependency contract failed before model download: "
        f"new_pip_conflicts={new_pip_conflicts} version_errors={version_errors} "
        f"import_smoke_rc={import_smoke.returncode}"
    )
print("P147 dependency contract passed")


section("5. Fail-fast installed Qwen3 runtime preflight before model download")
if INSTALL_COMPAT_UNSLOTH == "skip":
    raise RuntimeError("P147 strict runtime provenance cannot be skipped")
verification_code = r"""
import ast
import hashlib
import importlib.metadata
import inspect
import json
from pathlib import Path
import torch
import unsloth
import xformers
import xformers.ops
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments
import unsloth.models.qwen3 as qwen3_module

tree = ast.parse(inspect.getsource(qwen3_module))
called_flash_symbols = sorted({
    node.func.id
    for node in ast.walk(tree)
    if isinstance(node, ast.Call)
    and isinstance(node.func, ast.Name)
    and node.func.id.startswith("flash_attn")
})
missing_flash_symbols = [
    name for name in called_flash_symbols
    if not callable(getattr(qwen3_module, name, None))
]
report = {
    "torch_version": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "unsloth_file": inspect.getfile(unsloth),
    "qwen3_file": inspect.getfile(qwen3_module),
    "xformers_file": inspect.getfile(xformers),
    "unsloth_version": importlib.metadata.version("unsloth"),
    "xformers_version": importlib.metadata.version("xformers"),
    "qwen3_sha256": hashlib.sha256(Path(inspect.getfile(qwen3_module)).read_bytes()).hexdigest(),
    "called_flash_symbols": called_flash_symbols,
    "missing_flash_symbols": missing_flash_symbols,
    "xformers_attention_callable": callable(getattr(xformers.ops, "memory_efficient_attention", None)),
}
q = torch.randn((1, 8, 2, 16), device="cuda", dtype=torch.float16, requires_grad=True)
k = torch.randn((1, 8, 2, 16), device="cuda", dtype=torch.float16, requires_grad=True)
v = torch.randn((1, 8, 2, 16), device="cuda", dtype=torch.float16, requires_grad=True)
xformers_out = xformers.ops.memory_efficient_attention(q, k, v)
xformers_out.square().mean().backward()
torch.cuda.synchronize()
report["xformers_cuda_forward_backward"] = bool(
    xformers_out.is_cuda and q.grad is not None and k.grad is not None and v.grad is not None
)
print("P147_INSTALLED_RUNTIME_PREFLIGHT=" + json.dumps(report, sort_keys=True))
if torch.__version__ != "2.11.0+cu128" or torch.version.cuda != "12.8":
    raise RuntimeError(f"unexpected loaded Torch/CUDA identity: {torch.__version__} cuda={torch.version.cuda}")
if missing_flash_symbols:
    raise RuntimeError(f"installed Unsloth Qwen3 has unresolved attention symbols: {missing_flash_symbols}")
if not report["xformers_attention_callable"]:
    raise RuntimeError("xformers memory_efficient_attention is not callable")
if not report["xformers_cuda_forward_backward"]:
    raise RuntimeError("xformers CUDA forward/backward smoke did not produce gradients")
"""
preflight_process = run_streamed(
    [sys.executable, "-c", verification_code],
    check=True,
    label="verify_p147_installed_qwen3_runtime",
    timeout_seconds=600,
    idle_timeout_seconds=300,
)
preflight_prefix = "P147_INSTALLED_RUNTIME_PREFLIGHT="
preflight_rows = [
    line[len(preflight_prefix):]
    for line in preflight_process.stdout.splitlines()
    if line.startswith(preflight_prefix)
]
if len(preflight_rows) != 1:
    raise RuntimeError(
        f"installed runtime preflight emitted {len(preflight_rows)} machine-readable rows"
    )
P147_INSTALLED_RUNTIME_PREFLIGHT = json.loads(preflight_rows[0])
if not isinstance(P147_INSTALLED_RUNTIME_PREFLIGHT, dict):
    raise RuntimeError("installed runtime preflight payload is not an object")
os.environ["ARC_QWEN_STAGED_DEPENDENCY_PATH_MODE"] = "installed"
COLAB_COMPAT_UNSLOTH_REPORT = {
    "requested": INSTALL_COMPAT_UNSLOTH,
    "verification_returncode": preflight_process.returncode,
    "mode": "installed_runtime_contract",
    "output_tail": preflight_process.stdout[-12000:],
}
print("colab compatible unsloth runtime", json.dumps(COLAB_COMPAT_UNSLOTH_REPORT, sort_keys=True))
if HF_BRIDGE is not None:
    HF_BRIDGE.event("colab_compatible_unsloth_runtime", COLAB_COMPAT_UNSLOTH_REPORT, upload=True)


section("5a. Build and seal leakage-safe public training LOPO episodes")
PILOT_TASKS = int(PILOT_TASKS)
EPISODES_PER_TASK = int(EPISODES_PER_TASK)
OUTER_FOLDS = int(OUTER_FOLDS)
AUTOLEARN_SEED = int(AUTOLEARN_SEED)
if not 0 <= AUTOLEARN_SEED <= 2 ** 32 - 1:
    raise ValueError("AUTOLEARN_SEED must be between 0 and 2**32-1")
BOOTSTRAP_SAMPLES = int(BOOTSTRAP_SAMPLES)
if not OUTER_FOLDS <= PILOT_TASKS <= 1000:
    raise ValueError("PILOT_TASKS must be between OUTER_FOLDS and 1000 for P147")
if not 1 <= EPISODES_PER_TASK <= 4:
    raise ValueError("EPISODES_PER_TASK must be between 1 and 4")
if not 2 <= OUTER_FOLDS <= 5:
    raise ValueError("OUTER_FOLDS must be between 2 and 5")
if not 100 <= BOOTSTRAP_SAMPLES <= 10000:
    raise ValueError("BOOTSTRAP_SAMPLES must be between 100 and 10000")

AUTOLEARN_RUN_ID = f"p147_{RUN_ID}"
AUTOLEARN_ROOT = Path("/content/arc2_autolearning_runs") / AUTOLEARN_RUN_ID
EPISODE_DIR = AUTOLEARN_ROOT / "episodes"
RANKER_DIR = AUTOLEARN_ROOT / "ranker"
PROCESS_TRACE_PATH = AUTOLEARN_ROOT / "qwen_process_trace.jsonl"
AUTOLEARN_ROOT.mkdir(parents=True, exist_ok=False)
if PROCESS_TRACE_PATH.exists():
    raise FileExistsError(f"process trace already exists before generation: {PROCESS_TRACE_PATH}")
SOURCE_CHALLENGES = ARC2 / "data" / "arc-agi_training_challenges.json"
SOURCE_SOLUTIONS = ARC2 / "data" / "arc-agi_training_solutions.json"
episode_command = [
    sys.executable, "-u", str(ARC2 / "pipeline" / "autolearning_episode_builder.py"),
    "--challenges", str(SOURCE_CHALLENGES),
    "--solutions", str(SOURCE_SOLUTIONS),
    "--out-dir", str(EPISODE_DIR),
    "--task-limit", str(PILOT_TASKS),
    "--episodes-per-task", str(EPISODES_PER_TASK),
    "--folds", str(OUTER_FOLDS),
    "--seed", str(AUTOLEARN_SEED),
    "--protocol", EPISODE_PROTOCOL,
    "--source-partition", "official_training",
]
atomic_write_private_json(
    SOURCE_SOLUTIONS,
    json.loads(SOURCE_TRAINING_SOLUTIONS_PAYLOAD.decode("utf-8", errors="strict")),
)
try:
    run_streamed(
        episode_command,
        cwd=str(ARC2),
        env=sanitized_child_env(),
        check=True,
        label="build_lopo_episodes",
        timeout_seconds=600,
        idle_timeout_seconds=300,
    )
finally:
    SOURCE_SOLUTIONS.unlink(missing_ok=True)
if SOURCE_SOLUTIONS.exists():
    raise RuntimeError("source training solutions remained visible after LOPO construction")
LOPO_CHALLENGES = EPISODE_DIR / "lopo_challenges.json"
LOPO_SOLUTIONS = EPISODE_DIR / "lopo_solutions.json"
EPISODE_MANIFEST_PATH = EPISODE_DIR / "episode_manifest.json"
EPISODE_MANIFEST = json.loads(EPISODE_MANIFEST_PATH.read_text(encoding="utf-8"))

def canonical_json_sha256(path: Path) -> str:
    value = json.loads(Path(path).read_text(encoding="utf-8", errors="strict"))
    return canonical_json_payload_sha256(json.dumps(value).encode("utf-8"))


def canonical_json_payload_sha256(raw_payload: bytes) -> str:
    value = json.loads(raw_payload.decode("utf-8", errors="strict"))
    payload = json.dumps(
        value,
        ensure_ascii=False,
        separators=(",", ":"),
        sort_keys=True,
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def validate_episode_identity(
    manifest: dict,
    reference: dict,
    observed_hashes: dict,
) -> dict:
    field_map = {
        "source_challenges_sha256": "source_challenges_sha256",
        "source_solutions_sha256": "source_solutions_sha256",
        "episode_challenges_sha256": "episode_challenges_sha256",
        "episode_solutions_sha256": "episode_solutions_sha256",
        "episode_protocol": "protocol",
        "source_partition": "source_partition",
        "task_count": "task_count",
        "episode_count": "episode_count",
        "available_holdout_count": "available_holdout_count",
        "all_available_holdouts_included": "all_available_holdouts_included",
        "hidden_test_information_parity": "hidden_test_information_parity",
    }
    hash_fields = tuple(key for key in field_map if key.endswith("_sha256"))
    missing_reference = sorted(key for key in field_map if key not in reference)
    missing_manifest = sorted(field for field in field_map.values() if field not in manifest)
    missing_observed = sorted(key for key in hash_fields if key not in observed_hashes)
    if missing_reference or missing_manifest or missing_observed:
        raise RuntimeError(
            "P147 episode identity contract is incomplete before Qwen GPU work: "
            + json.dumps(
                {
                    "missing_manifest": missing_manifest,
                    "missing_observed_hashes": missing_observed,
                    "missing_reference": missing_reference,
                },
                sort_keys=True,
            )
        )

    identity = {
        reference_key: manifest[manifest_key]
        for reference_key, manifest_key in field_map.items()
    }
    mismatches = {}
    for key, actual in identity.items():
        expected = reference[key]
        if actual != expected:
            mismatches.setdefault(key, {}).update(
                {"expected": expected, "manifest": actual}
            )
    for key in hash_fields:
        observed = observed_hashes[key]
        expected = reference[key]
        manifest_value = identity[key]
        if observed != expected or observed != manifest_value:
            mismatches.setdefault(key, {}).update(
                {
                    "expected": expected,
                    "manifest": manifest_value,
                    "observed": observed,
                }
            )
    if mismatches:
        raise RuntimeError(
            "P147 episode identity mismatch before Qwen GPU work: "
            + json.dumps(mismatches, sort_keys=True)
        )
    return identity

EPISODE_OBSERVED_HASHES = {
    "source_challenges_sha256": canonical_json_sha256(SOURCE_CHALLENGES),
    "source_solutions_sha256": canonical_json_payload_sha256(SOURCE_TRAINING_SOLUTIONS_PAYLOAD),
    "episode_challenges_sha256": canonical_json_sha256(LOPO_CHALLENGES),
    "episode_solutions_sha256": canonical_json_sha256(LOPO_SOLUTIONS),
}
def seal_episode_identity(identity: dict) -> dict:
    if not isinstance(identity, dict) or not identity:
        raise RuntimeError("episode identity validator returned no evidence")
    if "episode_identity_sha256" in identity:
        raise RuntimeError("episode identity unexpectedly arrived pre-sealed")
    payload = json.dumps(
        identity, ensure_ascii=True, separators=(",", ":"), sort_keys=True
    ).encode("utf-8")
    sealed = dict(identity)
    sealed["episode_identity_sha256"] = hashlib.sha256(payload).hexdigest()
    return sealed


EPISODE_IDENTITY = seal_episode_identity(validate_episode_identity(
    EPISODE_MANIFEST,
    FROZEN_REFERENCE,
    EPISODE_OBSERVED_HASHES,
))

SOURCE_CHALLENGES_PAYLOAD = SOURCE_CHALLENGES.read_bytes()
SOURCE_SOLUTIONS_PAYLOAD = SOURCE_TRAINING_SOLUTIONS_PAYLOAD
EPISODE_MANIFEST_PAYLOAD = EPISODE_MANIFEST_PATH.read_bytes()
LOPO_SOLUTIONS_PAYLOAD = LOPO_SOLUTIONS.read_bytes()


def canonical_value_sha256(value) -> str:
    payload = json.dumps(
        value, ensure_ascii=False, separators=(",", ":"), sort_keys=True
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def validate_episode_manifest_semantics(manifest: dict) -> dict:
    from autolearning_episode_builder import fold_for_task

    expected_top_level = {
        "schema_version": "arc-agi-2-validation-episodes-v3",
        "seed": AUTOLEARN_SEED,
        "folds": OUTER_FOLDS,
        "task_limit": PILOT_TASKS,
        "episodes_per_task": EPISODES_PER_TASK,
        "protocol": EPISODE_PROTOCOL,
        "source_partition": "official_training",
        "label_boundary": "held public test outputs exist only in lopo_solutions.json until post-generation labeling",
        "information_parity": "original train demonstrations only",
    }
    mismatches = {
        key: {"expected": expected, "observed": manifest.get(key)}
        for key, expected in expected_top_level.items()
        if manifest.get(key) != expected
    }
    if mismatches:
        raise RuntimeError(f"episode manifest top-level mismatch: {mismatches}")
    records = manifest.get("records")
    if not isinstance(records, list) or len(records) != manifest.get("episode_count"):
        raise RuntimeError("episode manifest records are absent or incomplete")
    challenges = json.loads(LOPO_CHALLENGES.read_text(encoding="utf-8", errors="strict"))
    solutions = json.loads(LOPO_SOLUTIONS_PAYLOAD.decode("utf-8", errors="strict"))
    source_challenges = json.loads(SOURCE_CHALLENGES_PAYLOAD.decode("utf-8", errors="strict"))
    source_solutions = json.loads(SOURCE_SOLUTIONS_PAYLOAD.decode("utf-8", errors="strict"))
    episode_ids = [row.get("episode_id") for row in records if isinstance(row, dict)]
    if (
        len(episode_ids) != len(records)
        or len(set(episode_ids)) != len(episode_ids)
        or set(episode_ids) != set(challenges)
        or set(episode_ids) != set(solutions)
    ):
        raise RuntimeError("episode manifest record IDs do not bind exactly to challenge/solution keys")
    fold_counts: dict[str, int] = {}
    task_ids: set[str] = set()
    record_fields = {
        "episode_id", "task_id", "fold", "held_pair_index", "held_source",
        "pair_count", "visible_pair_count", "held_input_sha256",
        "held_output_sha256", "challenge_sha256", "solution_sha256",
    }
    for row in records:
        if set(row) != record_fields:
            raise RuntimeError(f"episode manifest record schema mismatch: {sorted(set(row) ^ record_fields)}")
        episode_id = row["episode_id"]
        task_id = row["task_id"]
        held_index = row["held_pair_index"]
        if task_id not in source_challenges or task_id not in source_solutions:
            raise RuntimeError(f"episode references unknown source task: {task_id!r}")
        task = source_challenges[task_id]
        task_solutions = source_solutions[task_id]
        if (
            not isinstance(held_index, int)
            or isinstance(held_index, bool)
            or not 0 <= held_index < len(task.get("test", []))
            or held_index >= len(task_solutions)
        ):
            raise RuntimeError(f"episode held index is invalid: {row}")
        expected_challenge = {
            "train": task["train"],
            "test": [{"input": task["test"][held_index]["input"]}],
        }
        expected_solution = [task_solutions[held_index]]
        expected_fold = fold_for_task(task_id, OUTER_FOLDS, AUTOLEARN_SEED)
        expected_row = {
            "fold": expected_fold,
            "held_source": "original_test",
            "pair_count": len(task["train"]) + len(task["test"]),
            "visible_pair_count": len(task["train"]),
            "held_input_sha256": canonical_value_sha256(task["test"][held_index]["input"]),
            "held_output_sha256": canonical_value_sha256(task_solutions[held_index]),
            "challenge_sha256": canonical_value_sha256(expected_challenge),
            "solution_sha256": canonical_value_sha256(expected_solution),
        }
        row_mismatches = {
            key: {"expected": expected, "observed": row.get(key)}
            for key, expected in expected_row.items()
            if row.get(key) != expected
        }
        if row_mismatches or challenges[episode_id] != expected_challenge or solutions[episode_id] != expected_solution:
            raise RuntimeError(
                f"episode record does not bind to source data: episode={episode_id} mismatches={row_mismatches}"
            )
        task_ids.add(task_id)
        fold_counts[str(expected_fold)] = fold_counts.get(str(expected_fold), 0) + 1
    if len(task_ids) != manifest.get("task_count") or len(task_ids) != PILOT_TASKS:
        raise RuntimeError("episode manifest task cardinality mismatch")
    if dict(sorted(fold_counts.items())) != manifest.get("fold_episode_counts"):
        raise RuntimeError("episode manifest fold counts mismatch")
    return {
        "manifest_sha256": canonical_value_sha256(manifest),
        "record_count": len(records),
        "task_count": len(task_ids),
        "fold_episode_counts": dict(sorted(fold_counts.items())),
    }


EPISODE_MANIFEST_SEMANTICS = validate_episode_manifest_semantics(EPISODE_MANIFEST)
EFFECTIVE_LOPO_KEYS_LIST = tuple(sorted(row["episode_id"] for row in EPISODE_MANIFEST["records"]))
EFFECTIVE_LOPO_KEYS = ",".join(EFFECTIVE_LOPO_KEYS_LIST)
RUN_KEYS_LIST = list(EFFECTIVE_LOPO_KEYS_LIST)
RUN_KEYS = EFFECTIVE_LOPO_KEYS
MAX_TASKS = len(EFFECTIVE_LOPO_KEYS_LIST)
LAB_PARAMETERS["effective_lopo_keys"] = EFFECTIVE_LOPO_KEYS
LAB_PARAMETERS["episode_manifest_sha256"] = EPISODE_MANIFEST_SEMANTICS["manifest_sha256"]
LAB_PARAMETERS["episode_manifest_raw_sha256"] = hashlib.sha256(EPISODE_MANIFEST_PAYLOAD).hexdigest()
LAB_PARAMETERS["effective_lopo_key_count"] = MAX_TASKS
LAB_PARAMETERS_SHA256 = seal_lab_parameters()
if HF_BRIDGE is not None:
    HF_BRIDGE.event(
        "lab_parameters_final",
        {"phase": "post_episode_identity", "parameters": LAB_PARAMETERS},
        upload=HF_BRIDGE.enabled,
    )
if EPISODE_MANIFEST["task_count"] != PILOT_TASKS:
    raise RuntimeError(f"task count mismatch: {EPISODE_MANIFEST['task_count']} != {PILOT_TASKS}")
if MAX_TASKS != EPISODE_MANIFEST["available_holdout_count"]:
    raise RuntimeError(
        f"holdout coverage mismatch: episodes={MAX_TASKS} "
        f"available={EPISODE_MANIFEST['available_holdout_count']}"
    )
if EPISODE_MANIFEST["all_available_holdouts_included"] is not True:
    raise RuntimeError("P147 must include every available original-test holdout")
if EPISODE_MANIFEST["hidden_test_information_parity"] is not True:
    raise RuntimeError("P147 episode construction lacks hidden-test information parity")
if any("output" in test for task in json.loads(LOPO_CHALLENGES.read_text(encoding="utf-8")).values() for test in task["test"]):
    raise RuntimeError("label leakage: held-out output found in LOPO challenges")
lopo_summary = {
    "run_id": AUTOLEARN_RUN_ID,
    "task_count": PILOT_TASKS,
    "episode_count": MAX_TASKS,
    "configured_run_keys": CONFIGURED_RUN_KEYS,
    "effective_lopo_keys": EFFECTIVE_LOPO_KEYS,
    "configured_keys_are_selection_input": False,
    "fold_episode_counts": EPISODE_MANIFEST["fold_episode_counts"],
    "challenge_sha256": EPISODE_MANIFEST["episode_challenges_sha256"],
    "solution_sha256": EPISODE_MANIFEST["episode_solutions_sha256"],
    "label_boundary": EPISODE_MANIFEST["label_boundary"],
    "identity": EPISODE_IDENTITY,
}
print("LOPO", json.dumps(lopo_summary, sort_keys=True))
if HF_BRIDGE is not None:
    HF_BRIDGE.event("lopo_episodes_ready", lopo_summary, upload=True)

LOPO_SOLUTIONS_PAYLOAD = LOPO_SOLUTIONS.read_bytes()
if canonical_json_sha256(LOPO_SOLUTIONS) != EPISODE_MANIFEST["episode_solutions_sha256"]:
    raise RuntimeError("LOPO solutions changed before label isolation")
LOPO_SOLUTIONS.unlink()
if LOPO_SOLUTIONS.exists():
    raise RuntimeError("LOPO solutions could not be removed before generation")
EPISODE_MANIFEST_PATH.unlink()
if EPISODE_MANIFEST_PATH.exists():
    raise RuntimeError("episode manifest could not be removed before generation")
solution_files = sorted((ARC2 / "data").glob("*solutions*.json"))
for solution_path in solution_files:
    if solution_path.is_symlink() or not solution_path.is_file():
        raise RuntimeError(f"solution source is not a regular file: {solution_path}")
    solution_path.unlink()
remaining_solution_files = sorted((ARC2 / "data").glob("*solutions*.json"))
if remaining_solution_files:
    raise RuntimeError(f"solution files remain visible before generation: {remaining_solution_files}")


section("5b. Stage official Kaggle Qwen model after all cheap gates")
# This downloads roughly 8-9 GB into the Colab runtime. It is reused by all profiles.
env = sanitized_child_env(allow={"KAGGLE_USERNAME", "KAGGLE_KEY"})
env.update({
    "PYTHONUNBUFFERED": "1",
    "PYTHONIOENCODING": "utf-8",
    "PYTHONUTF8": "1",
    "ARC_DOWNLOAD_QWEN_MODEL": "1",
    "ARC_DOWNLOAD_UNSLOTH_KERNEL": "0",
    "ARC_UPGRADE_KAGGLE_CLI": "0",
    "ARC_KAGGLE_OUTPUT_FILE_PATTERN": KAGGLE_OUTPUT_FILE_PATTERN,
    "ARC_KAGGLE_OUTPUT_PAGE_SIZE": "200",
    "ARC_KAGGLE_OUTPUT_MAX_EMPTY_FILTERED_PAGES": "180",
    "ARC_KAGGLE_OUTPUT_MAX_PAGES": "500",
    "ARC_UNSLOTH_DOWNLOAD_FALLBACK_CLI": "1",
    "ARC_PROBE_LOAD_TOKENIZER": "1",
    "ARC_PROBE_IMPORT_PACKAGES": "1",
    "ARC_PROBE_STRICT_FLASH_CAUSAL": FLASH_CAUSAL_STRICT,
})
stage_config = {
    "lab_config_version": LAB_CONFIG_VERSION,
    "experiment_id": EXPERIMENT_ID,
    "file_pattern": env["ARC_KAGGLE_OUTPUT_FILE_PATTERN"],
    "page_size": env["ARC_KAGGLE_OUTPUT_PAGE_SIZE"],
    "max_empty_filtered_pages": env["ARC_KAGGLE_OUTPUT_MAX_EMPTY_FILTERED_PAGES"],
    "max_pages": env["ARC_KAGGLE_OUTPUT_MAX_PAGES"],
    "unsloth_fallback_cli": env["ARC_UNSLOTH_DOWNLOAD_FALLBACK_CLI"],
    "unsloth_download_requested": env["ARC_DOWNLOAD_UNSLOTH_KERNEL"] == "1",
    "staged_dependency_path_mode": "installed_runtime_contract",
    "consumed_staged_artifact_count": 0,
    "colab_compat_unsloth_spec": COLAB_COMPAT_UNSLOTH_SPEC,
    "flash_causal_strict": FLASH_CAUSAL_STRICT,
}
if stage_config["unsloth_download_requested"] and stage_config["consumed_staged_artifact_count"] == 0:
    raise RuntimeError("P147 FinOps gate: refusing an Unsloth kernel download with zero consumed artifacts")
print("stage kaggle output config", json.dumps(stage_config, sort_keys=True))
if HF_BRIDGE is not None:
    HF_BRIDGE.event("stage_kaggle_output_config", stage_config, upload=True)
MODEL_ROOT = Path("/tmp/qwen3_4b_grids15_sft139")
if MODEL_ROOT.exists():
    shutil.rmtree(MODEL_ROOT)
env["ARC_QWEN_STAGE_DIR"] = str(MODEL_ROOT)
try:
    run_streamed(
        [sys.executable, "-u", str(ARC2 / "hf" / "hf_stage_kaggle_assets.py")],
        cwd=str(ARC2),
        env=env,
        check=True,
        label="stage_kaggle_qwen_unsloth",
        timeout_seconds=2400,
        idle_timeout_seconds=900,
    )
finally:
    (kaggle_dir / "kaggle.json").unlink(missing_ok=True)
    for credential_name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        os.environ.pop(credential_name, None)
        env.pop(credential_name, None)
MODEL_CANDIDATES = []
if MODEL_ROOT.is_dir():
    for config_path in MODEL_ROOT.rglob("config.json"):
        candidate_weights = sorted(config_path.parent.glob("*.safetensors"))
        if candidate_weights:
            MODEL_CANDIDATES.append((config_path.parent.resolve(), candidate_weights))
if len(MODEL_CANDIDATES) != 1:
    raise RuntimeError(f"expected exactly one staged model directory, found {len(MODEL_CANDIDATES)}")
MODEL_DIR, MODEL_WEIGHTS = MODEL_CANDIDATES[0]
MODEL_CONFIG = json.loads((MODEL_DIR / "config.json").read_text(encoding="utf-8", errors="strict"))
if MODEL_CONFIG.get("model_type") != "qwen3":
    raise RuntimeError(f"staged model_type is not qwen3: {MODEL_CONFIG.get('model_type')!r}")
architectures = MODEL_CONFIG.get("architectures")
if not isinstance(architectures, list) or "Qwen3ForCausalLM" not in architectures:
    raise RuntimeError(f"staged model architecture is not Qwen3ForCausalLM: {architectures!r}")
if any(path.is_symlink() for path in MODEL_DIR.rglob("*")):
    raise RuntimeError(f"staged model contains symlinks: {MODEL_DIR}")
MODEL_INDEXES = sorted(MODEL_DIR.glob("*.safetensors.index.json"))
if len(MODEL_INDEXES) > 1:
    raise RuntimeError(f"multiple safetensors indexes found: {MODEL_INDEXES}")
if MODEL_INDEXES:
    model_index = json.loads(MODEL_INDEXES[0].read_text(encoding="utf-8", errors="strict"))
    weight_map = model_index.get("weight_map")
    if not isinstance(weight_map, dict) or not weight_map:
        raise RuntimeError("safetensors index has no non-empty weight_map")
    indexed_shards = {str(name) for name in weight_map.values()}
    actual_shards = {path.name for path in MODEL_WEIGHTS}
    if indexed_shards != actual_shards:
        raise RuntimeError(
            f"safetensors shard/index mismatch: indexed={sorted(indexed_shards)} actual={sorted(actual_shards)}"
        )
elif len(MODEL_WEIGHTS) != 1:
    raise RuntimeError("multiple safetensors shards require exactly one index")
MODEL_FILES = [path for path in MODEL_DIR.rglob("*") if path.is_file()]
EXPECTED_MODEL_FILE_SIZES = {
    "added_tokens.json": 68,
    "config.json": 1532,
    "generation_config.json": 113,
    "model-00001-of-00002.safetensors": 4_996_836_472,
    "model-00002-of-00002.safetensors": 2_270_397_024,
    "model.safetensors.index.json": 32_913,
    "special_tokens_map.json": 367,
    "tokenizer.json": 1_731,
    "tokenizer_config.json": 988,
    "vocab.json": 94,
}
OBSERVED_MODEL_FILE_SIZES = {
    path.relative_to(MODEL_DIR).as_posix(): path.stat().st_size
    for path in MODEL_FILES
}
if OBSERVED_MODEL_FILE_SIZES != EXPECTED_MODEL_FILE_SIZES:
    raise RuntimeError(
        "staged model inventory does not match Kaggle immutable version metadata: "
        f"observed={OBSERVED_MODEL_FILE_SIZES} expected={EXPECTED_MODEL_FILE_SIZES}"
    )
MODEL_TOTAL_BYTES = sum(path.stat().st_size for path in MODEL_FILES)
if not MODEL_FILES or MODEL_TOTAL_BYTES < 1_000_000_000:
    raise RuntimeError(
        f"staged model is incomplete: dir={MODEL_DIR} files={len(MODEL_FILES)} bytes={MODEL_TOTAL_BYTES}"
    )
from safetensors import safe_open
MODEL_TENSOR_COUNT = 0
for weight_path in MODEL_WEIGHTS:
    with safe_open(str(weight_path), framework="pt", device="cpu") as handle:
        keys = list(handle.keys())
        if not keys:
            raise RuntimeError(f"safetensors shard has no tensors: {weight_path}")
        MODEL_TENSOR_COUNT += len(keys)
stage_config["model_file_count"] = len(MODEL_FILES)
stage_config["model_total_bytes"] = MODEL_TOTAL_BYTES
stage_config["model_weight_count"] = len(MODEL_WEIGHTS)
stage_config["model_tensor_count"] = MODEL_TENSOR_COUNT
stage_config["model_type"] = MODEL_CONFIG.get("model_type")
stage_config["model_architectures"] = MODEL_CONFIG.get("architectures")
stage_config["model_weight_sha256"] = {
    path.name: sha256_file(path) for path in MODEL_WEIGHTS
}
stage_config["model_file_sha256"] = {
    path.relative_to(MODEL_DIR).as_posix(): sha256_file(path)
    for path in MODEL_FILES
}
stage_config["model_version"] = "sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1"
stage_config["model_expected_file_sizes"] = EXPECTED_MODEL_FILE_SIZES
stage_config["model_observed_file_sizes"] = OBSERVED_MODEL_FILE_SIZES
stage_config["remote_sha256_available"] = False
stage_config["resolved_model_dir"] = str(MODEL_DIR)
print("staging complete", json.dumps(stage_config, sort_keys=True))
if HF_BRIDGE is not None:
    HF_BRIDGE.event("model_staging_complete", stage_config, upload=True)


section("6. Run frozen Qwen control over LOPO episodes")
if len(PROFILES) != 1:
    raise RuntimeError("P147 label-isolated protocol requires exactly one outer profile")


def run_profile(profile: str) -> dict:
    run_id = f"colab_{profile}_{time.time_ns()}"
    label_dir = Path(tempfile.mkdtemp(prefix=f"p147-label-{run_id}-", dir="/tmp"))
    os.chmod(label_dir, 0o700)
    ephemeral_solutions = label_dir / "solutions.json"
    ephemeral_manifest = label_dir / "episode_manifest.json"
    atomic_write_private_json(
        ephemeral_solutions,
        json.loads(LOPO_SOLUTIONS_PAYLOAD.decode("utf-8", errors="strict")),
    )
    atomic_write_bytes(ephemeral_manifest, EPISODE_MANIFEST_PAYLOAD)
    profile_env = sanitized_child_env()
    profile_env.update({
        "PYTHONUNBUFFERED": "1",
        "PYTHONIOENCODING": "utf-8",
        "PYTHONUTF8": "1",
        "ARC_QWEN_PROFILE": profile,
        "ARC_QWEN_RUN_TAG": f"colab-{profile}",
        "ARC_QWEN_GPUS": FORCE_GPU_COUNT,
        "ARC_QWEN_THROUGHPUT_GPUS": FORCE_GPU_COUNT,
        "ARC_QWEN_THROUGHPUT_RUN_ID": run_id,
        "ARC_QWEN_THROUGHPUT_KEYS": RUN_KEYS,
        "ARC_QWEN_THROUGHPUT_CHALLENGES": str(LOPO_CHALLENGES),
        "ARC_QWEN_THROUGHPUT_SOLUTIONS": str(ephemeral_solutions),
        "ARC_AUTOLEARN_EPHEMERAL_LABELS": "1",
        "ARC_QWEN_PROCESS_TRACE": str(PROCESS_TRACE_PATH),
        "ARC_QWEN_TRACE_BATCHES": "1" if ENABLE_FULL_PROCESS_TRACE else "0",
        "ARC_QWEN_TRACE_FILES": "1" if ENABLE_FULL_PROCESS_TRACE else "0",
        "ARC_AUTOLEARN_RUN_ID": AUTOLEARN_RUN_ID,
        "ARC_AUTOLEARN_EPISODE_MANIFEST": str(ephemeral_manifest),
        "ARC_AUTOLEARN_EPHEMERAL_MANIFEST": "1",
        "ARC_AUTOLEARN_EPISODE_MANIFEST_SHA256": LAB_PARAMETERS["episode_manifest_raw_sha256"],
        "ARC_AUTOLEARN_LABEL_BOUNDARY": "post_generation_only",
        "ARC_QWEN_THROUGHPUT_MAX_TASKS": str(MAX_TASKS),
        "ARC_QWEN_THROUGHPUT_SECONDS": str(SECONDS_PER_PROFILE),
        "ARC_QWEN_TASK_ORDER": "complexity_desc",
        "ARC_QWEN_MODEL_DIR": str(MODEL_DIR),
        "ARC_SELECTOR_WEIGHTS": SELECTOR_WEIGHT_SPEC,
        "ARC_QWEN_SELECTOR_SWEEP": "1" if SELECTOR_SWEEP_ENABLED else "0",
        "ARC_QWEN_REQUIRE_LABELED_ANALYSIS": "1",
        "ARC_QWEN_SELECTOR_SWEEP_MODES": SELECTOR_SWEEP_MODES,
        "ARC_MAX_DUPLICATE_ATTEMPT_RATE": str(MAX_DUPLICATE_ATTEMPT_RATE),
        "ARC_QWEN_THROUGHPUT_REQUIRE_PROBE": "1",
        "ARC_QWEN_THROUGHPUT_SKIP_PROBE": "0",
        "ARC_QWEN_THROUGHPUT_ALLOW_PARTIAL": "0",
        "ARC_QWEN_STRICT": "1",
        "ARC_QWEN_STAGED_DEPENDENCY_PATH_MODE": "installed",
        "ARC_QWEN_INSTALLED_DEPENDENCY_MODULES_JSON": '["unsloth", "xformers"]',
        "ARC_QWEN_THROUGHPUT_FAIL_ON_INVALID_CANDIDATE_FILES": "1",
        "ARC_QWEN_THROUGHPUT_USE_SYMBOLIC": "1" if USE_SYMBOLIC else "0",
        "ARC_MISSING_SYMBOLIC_FALLBACK": "1" if MISSING_SYMBOLIC_FALLBACK else "0",
        "ARC_PROBE_LOAD_TOKENIZER": "1",
        "ARC_PROBE_IMPORT_PACKAGES": "1",
        "ARC_PROBE_RECURSIVE_SEARCH": "1",
        "ARC_PROBE_STRICT_FLASH_CAUSAL": FLASH_CAUSAL_STRICT,
        "ARC_EXPERIMENT_ID": EXPERIMENT_ID,
        "ARC_EXPERIMENT_NOTE": EXPERIMENT_NOTE,
    })
    profile_env.update(QWEN_OPTIONAL_OVERRIDES)
    try:
        completed = run_streamed(
            [sys.executable, "-u", str(ARC2 / "hf" / "qwen_worker_throughput.py")],
            cwd=str(ARC2),
            env=profile_env,
            check=False,
            label=f"qwen_throughput_{profile}",
            timeout_seconds=SECONDS_PER_PROFILE + 1800,
            idle_timeout_seconds=1800,
        )
        if ephemeral_solutions.exists():
            raise RuntimeError("Qwen orchestrator did not consume/remove the ephemeral label file")
        if not ephemeral_manifest.is_file():
            raise RuntimeError("Qwen orchestrator did not restore the ephemeral manifest")
        if ephemeral_manifest.read_bytes() != EPISODE_MANIFEST_PAYLOAD:
            raise RuntimeError("Qwen orchestrator restored a mutated episode manifest")
    finally:
        shutil.rmtree(label_dir, ignore_errors=True)
    rc = completed.returncode
    report_path = Path("/tmp/arc_qwen_throughput") / run_id / "qwen_throughput_report.json"
    if not report_path.is_file():
        return {
            "status": "error",
            "profile": profile,
            "process_returncode": rc,
            "worker_returncode": None,
            "postprocess_returncode": None,
            "format_ok": False,
            "expected_outputs": MAX_TASKS,
            "coverage": {"outputs_with_qwen_candidates": 0},
            "report_path": str(report_path),
            "report_missing": True,
            "process_output_tail": (completed.stdout or "")[-12000:],
        }
    report = json.loads(report_path.read_text(encoding="utf-8", errors="strict"))
    report["process_returncode"] = rc
    report["report_path"] = str(report_path)
    return report


def _exact_zero_returncode(report: dict, key: str) -> bool:
    value = report.get(key)
    return isinstance(value, int) and not isinstance(value, bool) and value == 0


REQUIRED_PROFILE_ARTIFACT_KEYS = (
    "submission", "manifest", "preflight", "candidate_eval", "diagnostics",
    "selector_sweep", "portfolio_report", "portfolio_seed_analysis",
)


def validated_report_workdir(report: dict) -> Path:
    run_id = report.get("run_id")
    work_value = report.get("work")
    if (
        not isinstance(run_id, str)
        or run_id in {".", ".."}
        or re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,127}", run_id) is None
    ):
        raise RuntimeError(f"unsafe report run_id: {run_id!r}")
    if not isinstance(work_value, str) or not work_value.strip():
        raise RuntimeError("report workdir is missing")
    sealed_root = Path("/tmp/arc_qwen_throughput").resolve(strict=True)
    expected = (sealed_root / run_id).resolve(strict=True)
    observed = Path(work_value).resolve(strict=True)
    if sealed_root not in expected.parents or observed != expected or not observed.is_dir():
        raise RuntimeError(f"report workdir escaped sealed root: observed={observed} expected={expected}")
    return observed


def required_report_artifact_paths(report: dict) -> dict[str, Path]:
    workdir = validated_report_workdir(report)
    artifact_map = report.get("artifacts")
    if not isinstance(artifact_map, dict):
        raise RuntimeError("report artifacts mapping is missing")
    raw_paths = {"throughput_report": report.get("report_path")}
    raw_paths.update({key: artifact_map.get(key) for key in REQUIRED_PROFILE_ARTIFACT_KEYS})
    resolved: dict[str, Path] = {}
    for label, raw_path in raw_paths.items():
        if not isinstance(raw_path, str) or not raw_path.strip():
            raise RuntimeError(f"required report artifact path is missing: {label}")
        path = Path(raw_path).resolve(strict=True)
        if workdir not in path.parents or not path.is_file() or path.is_symlink() or path.stat().st_size <= 0:
            raise RuntimeError(f"required report artifact is invalid: label={label} path={path}")
        resolved[label] = path
    if len(set(resolved.values())) != len(resolved):
        raise RuntimeError(f"required report artifact paths collide: {resolved}")
    return resolved


def profile_is_clean(report: dict) -> bool:
    coverage = report.get("coverage") if isinstance(report.get("coverage"), dict) else {}
    firewall = (
        report.get("generation_label_firewall")
        if isinstance(report.get("generation_label_firewall"), dict)
        else {}
    )
    expected = report.get("expected_outputs")
    covered = coverage.get("outputs_with_qwen_candidates", coverage.get("covered_outputs"))
    try:
        required_report_artifact_paths(report)
    except (OSError, RuntimeError, ValueError):
        return False
    return (
        report.get("status") == "ok"
        and all(_exact_zero_returncode(report, key) for key in (
            "process_returncode", "probe_returncode", "worker_returncode", "postprocess_returncode"
        ))
        and report.get("probe_required") is True
        and report.get("probe_executed") is True
        and report.get("strict_mode") is True
        and report.get("allow_partial") is False
        and report.get("strict_ok") is True
        and report.get("ephemeral_labels_consumed_before_worker") is True
        and report.get("ephemeral_manifest_requested") is True
        and report.get("episode_manifest_sha256_verified") is True
        and report.get("ephemeral_manifest_removed_before_worker") is True
        and report.get("ephemeral_manifest_restored_after_worker") is True
        and firewall.get("solutions_materialized_after_worker") is True
        and firewall.get("ephemeral_source_removed_before_worker") is True
        and firewall.get("solution_path_passed_to_worker") is False
        and firewall.get("episode_manifest_path_passed_to_worker") is False
        and firewall.get("ephemeral_manifest_sha256_verified") is True
        and firewall.get("ephemeral_manifest_removed_before_worker") is True
        and firewall.get("ephemeral_manifest_restored_after_worker") is True
        and report.get("format_ok") is True
        and report.get("portfolio_complete") is True
        and report.get("selector_sweep_ok") is True
        and report.get("portfolio_analysis_ok") is True
        and report.get("labeled_analysis_ok") is True
        and isinstance(expected, int) and not isinstance(expected, bool)
        and expected == MAX_TASKS and expected > 0
        and isinstance(covered, int) and not isinstance(covered, bool)
        and covered == expected
    )


reports = []
for profile in PROFILES:
    section(f"RUN PROFILE {profile}")
    report = run_profile(profile)
    reports.append(report)
    if profile == PROFILES[0] and STOP_AFTER_BASELINE_FAILURE and not profile_is_clean(report):
        payload = {
            "profile": profile,
            "status": report.get("status"),
            "process_returncode": report.get("process_returncode"),
            "probe_returncode": report.get("probe_returncode"),
            "worker_returncode": report.get("worker_returncode"),
            "postprocess_returncode": report.get("postprocess_returncode"),
            "format_ok": report.get("format_ok"),
            "coverage": report.get("coverage"),
            "candidate_count": report.get("candidate_count"),
            "selector_score": report.get("selector_score"),
            "oracle_score": report.get("oracle_score"),
            "reason": "baseline_not_clean_stop_requested",
        }
        print("baseline failed clean gate; stopping remaining profiles", json.dumps(payload, sort_keys=True))
        if HF_BRIDGE is not None:
            HF_BRIDGE.event("baseline_clean_gate_failed_stop_remaining_profiles", payload, upload=True)
        break


unclean_profiles = [str(report.get("profile")) for report in reports if not profile_is_clean(report)]
if unclean_profiles:
    raise RuntimeError(f"Qwen profiles failed the sealed clean gate: {unclean_profiles}")
if EPISODE_MANIFEST_PATH.exists():
    raise RuntimeError("canonical episode manifest unexpectedly materialized during generation")
restored_manifest = json.loads(EPISODE_MANIFEST_PAYLOAD.decode("utf-8", errors="strict"))
restored_manifest_semantics = validate_episode_manifest_semantics(restored_manifest)
if restored_manifest_semantics != EPISODE_MANIFEST_SEMANTICS:
    raise RuntimeError(
        f"episode manifest semantics changed across generation: {restored_manifest_semantics}"
    )
atomic_write_bytes(EPISODE_MANIFEST_PATH, EPISODE_MANIFEST_PAYLOAD)
if EPISODE_MANIFEST_PATH.read_bytes() != EPISODE_MANIFEST_PAYLOAD:
    raise RuntimeError("episode manifest failed exact post-generation restoration")
atomic_write_private_json(
    LOPO_SOLUTIONS,
    json.loads(LOPO_SOLUTIONS_PAYLOAD.decode("utf-8", errors="strict")),
)
if canonical_json_sha256(LOPO_SOLUTIONS) != EPISODE_MANIFEST["episode_solutions_sha256"]:
    raise RuntimeError("LOPO solutions changed while crossing the post-generation label boundary")


section("7. Summarize, gate, and persist reports to Drive")
sys.path.insert(0, str(ARC2 / "pipeline"))
from generation_candidate_decision import decide_generation_candidate


def pick(report: dict, *path: str, default=None):
    cur = report
    for key in path:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key)
    return default if cur is None else cur


rows = []
for report in reports:
    rows.append({
        "profile": report.get("profile"),
        "status": report.get("status"),
        "rc": report.get("process_returncode"),
        "worker_elapsed_s": report.get("worker_elapsed_s"),
        "candidate_count": report.get("candidate_count"),
        "covered": pick(report, "coverage", "outputs_with_qwen_candidates"),
        "expected": report.get("expected_outputs"),
        "unique_candidate_grids_median": pick(report, "candidate_diversity", "unique_candidate_grids_median"),
        "one_unique_candidate_outputs": pick(report, "candidate_diversity", "one_unique_candidate_outputs"),
        "attempt2_input_fallback_outputs": pick(report, "candidate_diversity", "attempt2_input_fallback_outputs"),
        "selector_score": report.get("selector_score"),
        "oracle_score": report.get("oracle_score"),
        "recoverable_selector_gap": report.get("recoverable_selector_gap"),
        "selector_sweep_best": pick(report, "selector_sweep_best", "name"),
        "selector_sweep_best_score": pick(report, "selector_sweep_best", "selector_score"),
        "portfolio_status": pick(report, "portfolio", "status"),
        "portfolio_completed_runs": pick(report, "portfolio", "summary", "completed_runs"),
        "portfolio_oracle_overlap": pick(report, "portfolio_seed_analysis", "oracle_overlap"),
        "portfolio_order_sensitivity": pick(report, "portfolio_seed_analysis", "order_sensitivity"),
        "estimated_hours_for_259_outputs": report.get("estimated_hours_for_259_outputs"),
        "report_path": report.get("report_path"),
    })
print(json.dumps(rows, indent=2))

decision_json = {
    "status": "not_applicable",
    "action": "use_generation_candidate_decision",
    "reason": "P147 has one outer profile; seed-a and seed-b are internal portfolio runs",
    "official_kaggle_score_claim": None,
}
print("\nOUTER A/B DECISION")
print(json.dumps(decision_json, indent=2))

results_root = (
    Path("/content/drive/MyDrive/arc2016_colab_results")
    if DRIVE_AVAILABLE
    else Path("/content/arc2016_colab_results")
)
out_dir = results_root / RUN_ID
out_dir.mkdir(parents=True, exist_ok=False)
summary_path = out_dir / "summary.json"
decision_path = out_dir / "qwen_ab_decision.json"
lab_parameters_path = out_dir / "lab_parameters.json"
atomic_write_text(summary_path, json.dumps(rows, indent=2))
atomic_write_text(decision_path, json.dumps(decision_json, indent=2))
atomic_write_text(lab_parameters_path, json.dumps(LAB_PARAMETERS, indent=2, sort_keys=True))
artifact_paths = [summary_path, decision_path, lab_parameters_path, DEPENDENCY_CONTRACT_PATH]
for report in reports:
    profile = str(report.get("profile") or "profile")
    for label, src in required_report_artifact_paths(report).items():
        suffix = src.suffix or ".json"
        dst = out_dir / f"{profile}_{label}{suffix}"
        atomic_write_bytes(dst, src.read_bytes())
        artifact_paths.append(dst)

GENERATION_EVIDENCE_SCOPE = "public_training_lopo_proxy_not_kaggle_score"
EXPERIMENT_DESIGN = "dual_seed_candidate_portfolio_noncausal"
CAUSAL_ATTRIBUTION_ALLOWED = False
generation_decision = decide_generation_candidate(
    reports[0].get("portfolio_seed_analysis") if reports else None,
    reports[0] if reports else None,
    control_tag="seed-a",
    candidate_tag="seed-b",
    min_outputs=FROZEN_P137_REFERENCE["outputs_total"],
    max_attempt2_input_fallback_rate=MAX_ATTEMPT2_INPUT_FALLBACK_RATE,
    reference_control=FROZEN_P137_REFERENCE,
    current_evidence={
        "source_challenges_sha256": EPISODE_MANIFEST["source_challenges_sha256"],
        "source_solutions_sha256": EPISODE_MANIFEST["source_solutions_sha256"],
        "episode_challenges_sha256": EPISODE_MANIFEST["episode_challenges_sha256"],
        "episode_solutions_sha256": EPISODE_MANIFEST["episode_solutions_sha256"],
        "episode_protocol": EPISODE_MANIFEST["protocol"],
        "source_partition": EPISODE_MANIFEST["source_partition"],
        "task_count": EPISODE_MANIFEST["task_count"],
        "episode_count": EPISODE_MANIFEST["episode_count"],
        "available_holdout_count": EPISODE_MANIFEST["available_holdout_count"],
        "all_available_holdouts_included": EPISODE_MANIFEST["all_available_holdouts_included"],
        "hidden_test_information_parity": EPISODE_MANIFEST["hidden_test_information_parity"],
        "selector_spec": SELECTOR_WEIGHT_SPEC,
        "profile": PROFILES[0] if len(PROFILES) == 1 else None,
        "model_asset_id": "qwen3_4b_grids15_sft139",
        "total_budget_seconds": SECONDS_PER_PROFILE,
    },
)
generation_decision_json = generation_decision.to_dict()
allowed_generation_statuses = {
    "rejected", "no_promotion", "promising_diagnostic",
    "generator_gain_selector_gap", "candidate",
}
generation_status = generation_decision_json.get("status")
if generation_status not in allowed_generation_statuses:
    raise RuntimeError(
        f"generation decision is blocked or unknown: status={generation_status!r} "
        f"reasons={generation_decision_json.get('reasons')}"
    )
generation_decision_path = out_dir / "generation_candidate_decision.json"
atomic_write_text(generation_decision_path, json.dumps(generation_decision_json, indent=2, sort_keys=True))
artifact_paths.append(generation_decision_path)
print("GENERATION CANDIDATE DECISION")
print(json.dumps(generation_decision_json, indent=2, sort_keys=True))
if HF_BRIDGE is not None:
    HF_BRIDGE.event("generation_candidate_decision", generation_decision_json, upload=True)

print("saved", out_dir)


section("8. Post-generation label join, cross-fit predictor, and selector replay")
if not reports:
    raise RuntimeError("no Qwen control report exists")
control_report = reports[0]
control_artifacts = control_report.get("artifacts") if isinstance(control_report.get("artifacts"), dict) else {}
if not profile_is_clean(control_report):
    failure_payload = {
        "status": "autolearning_skipped",
        "reason": "qwen_control_not_clean",
        "worker_returncode": control_report.get("worker_returncode"),
        "postprocess_returncode": control_report.get("postprocess_returncode"),
        "format_ok": control_report.get("format_ok"),
        "coverage": control_report.get("coverage"),
        "candidate_count": control_report.get("candidate_count"),
        "rule": "never train or evaluate a ranker on incomplete generator evidence",
    }
    failure_path = AUTOLEARN_ROOT / "autolearning_skipped_unclean_control.json"
    atomic_write_text(failure_path, json.dumps(failure_payload, indent=2, sort_keys=True))
    failure_output_dir = out_dir / "autolearning_failure"
    failure_output_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(failure_path, failure_output_dir / failure_path.name)
    if PROCESS_TRACE_PATH.is_file():
        shutil.copy2(PROCESS_TRACE_PATH, failure_output_dir / PROCESS_TRACE_PATH.name)
    failure_archive = Path(shutil.make_archive(
        str(AUTOLEARN_ROOT.parent / f"{AUTOLEARN_RUN_ID}_failed_artifacts"),
        "zip",
        root_dir=AUTOLEARN_ROOT,
    ))
    failure_drive_archive = failure_output_dir / failure_archive.name
    shutil.copy2(failure_archive, failure_drive_archive)
    artifact_paths.extend([failure_path, failure_archive, failure_drive_archive])
    if HF_BRIDGE is not None:
        HF_BRIDGE.event("autolearning_skipped_unclean_control", failure_payload, upload=True)
        HF_BRIDGE.sync_once(extra_paths=artifact_paths)
    if DRIVE_LOG_MIRROR is not None:
        DRIVE_LOG_MIRROR.sync_once()
    raise RuntimeError(
        "P147 stopped before autolearning because the Qwen control was not clean; "
        "diagnostics were sealed before exit"
    )
CANDIDATE_MANIFEST_PATH = Path(str(control_artifacts.get("manifest") or ""))
if not CANDIDATE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"candidate manifest missing: {CANDIDATE_MANIFEST_PATH}")

try:
    import sklearn
    sklearn_version = sklearn.__version__
except ImportError as exc:
    raise RuntimeError("scikit-learn disappeared after the P147 dependency contract") from exc

ranker_command = [
    sys.executable, "-u", str(ARC2 / "pipeline" / "autolearning_ranker.py"),
    "--challenges", str(LOPO_CHALLENGES),
    "--solutions", str(LOPO_SOLUTIONS),
    "--episode-manifest", str(EPISODE_MANIFEST_PATH),
    "--candidates", str(CANDIDATE_MANIFEST_PATH),
    "--process-trace", str(PROCESS_TRACE_PATH),
    "--out-dir", str(RANKER_DIR),
    "--bootstrap-samples", str(BOOTSTRAP_SAMPLES),
    "--seed", str(AUTOLEARN_SEED),
    "--min-comparable-tasks", str(PILOT_TASKS),
    "--selector-weights", SELECTOR_WEIGHT_SPEC,
]
if EXPECTED_QWEN_RUN_TAGS:
    ranker_command.extend(["--expected-run-tags", ",".join(EXPECTED_QWEN_RUN_TAGS)])
ranker_process = run_streamed(
    ranker_command, cwd=str(ARC2 / "pipeline"), env=sanitized_child_env(), check=False, label="crossfit_autolearning_ranker", timeout_seconds=1800, idle_timeout_seconds=600
)
if ranker_process.returncode != 0:
    failure_path = AUTOLEARN_ROOT / "autolearning_failure.json"
    atomic_write_text(failure_path, json.dumps({
        "status": "failed",
        "returncode": ranker_process.returncode,
        "candidate_manifest": str(CANDIDATE_MANIFEST_PATH),
        "process_trace": str(PROCESS_TRACE_PATH),
        "output_tail": (ranker_process.stdout or "")[-12000:],
        "rule": "fail closed; no learned selector is promoted",
    }, indent=2))
    artifact_paths.append(failure_path)
    if HF_BRIDGE is not None:
        HF_BRIDGE.event("autolearning_failed", json.loads(failure_path.read_text(encoding="utf-8")), upload=True)
        HF_BRIDGE.sync_once(extra_paths=artifact_paths, wait_for_lock=True)
    if DRIVE_LOG_MIRROR is not None:
        DRIVE_LOG_MIRROR.sync_once(wait_for_lock=True)
    raise RuntimeError(
        f"autolearning ranker failed rc={ranker_process.returncode}; "
        f"output_tail={(ranker_process.stdout or '')[-4000:]}"
    )
autolearning_report_path = RANKER_DIR / "autolearning_report.json"
if not autolearning_report_path.is_file():
    raise FileNotFoundError(f"ranker returned zero but report is missing: {autolearning_report_path}")
AUTOLEARNING_REPORT = json.loads(autolearning_report_path.read_text(encoding="utf-8", errors="strict"))
AUTOLEARNING_REPORT["sklearn_version"] = sklearn_version
AUTOLEARNING_REPORT["qwen_control_status"] = control_report.get("status")
AUTOLEARNING_REPORT["official_kaggle_score_claim"] = None
autolearning_coverage_errors = []
if AUTOLEARNING_REPORT.get("full_oof_coverage") is not True:
    autolearning_coverage_errors.append("full_oof_coverage is not true")
if AUTOLEARNING_REPORT.get("comparable_oof_episodes") != MAX_TASKS:
    autolearning_coverage_errors.append(
        f"comparable_oof_episodes={AUTOLEARNING_REPORT.get('comparable_oof_episodes')} expected={MAX_TASKS}"
    )
if AUTOLEARNING_REPORT.get("distinct_comparable_tasks") != PILOT_TASKS:
    autolearning_coverage_errors.append(
        f"distinct_comparable_tasks={AUTOLEARNING_REPORT.get('distinct_comparable_tasks')} expected={PILOT_TASKS}"
    )
if AUTOLEARNING_REPORT.get("trained_folds") != AUTOLEARNING_REPORT.get("expected_folds"):
    autolearning_coverage_errors.append("trained_folds differ from expected_folds")
if AUTOLEARNING_REPORT.get("minimum_sample_met") is not True:
    autolearning_coverage_errors.append("minimum_sample_met is not true")
process_trace_coverage = AUTOLEARNING_REPORT.get("process_trace_coverage")
if not isinstance(process_trace_coverage, dict):
    autolearning_coverage_errors.append("process_trace_coverage is missing")
else:
    if process_trace_coverage.get("required") is not True:
        autolearning_coverage_errors.append("process trace was not required by the ranker")
    if process_trace_coverage.get("file_present") is not True:
        autolearning_coverage_errors.append("process trace file was not present")
    if process_trace_coverage.get("complete") is not True:
        autolearning_coverage_errors.append(
            "process trace is incomplete: "
            f"missing={process_trace_coverage.get('missing_episode_ids')} "
            f"unknown={process_trace_coverage.get('unknown_episode_ids')} "
            f"incomplete={process_trace_coverage.get('incomplete_episode_ids')} "
            f"missing_run_tags={process_trace_coverage.get('missing_run_tags_by_episode')} "
            f"incomplete_run_tags={process_trace_coverage.get('incomplete_run_tags_by_episode')}"
        )
    if process_trace_coverage.get("expected_run_tags") != EXPECTED_QWEN_RUN_TAGS:
        autolearning_coverage_errors.append(
            f"process trace run-tag contract mismatch: observed={process_trace_coverage.get('expected_run_tags')} "
            f"expected={EXPECTED_QWEN_RUN_TAGS}"
        )
    if process_trace_coverage.get("expected_episode_count") != MAX_TASKS:
        autolearning_coverage_errors.append(
            f"process trace expected_episode_count={process_trace_coverage.get('expected_episode_count')} expected={MAX_TASKS}"
        )
    if process_trace_coverage.get("completed_episode_count") != MAX_TASKS:
        autolearning_coverage_errors.append(
            f"process trace completed_episode_count={process_trace_coverage.get('completed_episode_count')} expected={MAX_TASKS}"
        )
AUTOLEARNING_REPORT["notebook_coverage_gate_errors"] = autolearning_coverage_errors
atomic_write_text(autolearning_report_path, json.dumps(AUTOLEARNING_REPORT, indent=2, sort_keys=True))
if autolearning_coverage_errors:
    raise RuntimeError(f"autolearning coverage gate failed: {autolearning_coverage_errors}")
print("AUTOLEARNING REPORT")
print(json.dumps(AUTOLEARNING_REPORT, indent=2, sort_keys=True))
if HF_BRIDGE is not None:
    HF_BRIDGE.event("autolearning_ranker_validated", AUTOLEARNING_REPORT, upload=True)

control_work_value = control_report.get("work")
control_run_id = control_report.get("run_id")
if not isinstance(control_work_value, str) or not control_work_value.strip():
    raise RuntimeError("Qwen control report has no non-empty workdir")
if not isinstance(control_run_id, str) or not control_run_id.strip():
    raise RuntimeError("Qwen control report has no non-empty run_id")
if (
    control_run_id in {".", ".."}
    or re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,127}", control_run_id) is None
):
    raise RuntimeError(f"Qwen control report has unsafe run_id: {control_run_id!r}")
control_work = Path(control_work_value).resolve(strict=True)
sealed_work_root = Path("/tmp/arc_qwen_throughput").resolve(strict=True)
expected_control_work = (sealed_work_root / control_run_id).resolve(strict=True)
if (
    sealed_work_root not in expected_control_work.parents
    or control_work != expected_control_work
    or not control_work.is_dir()
):
    raise RuntimeError(
        f"Qwen control workdir escaped its sealed root: observed={control_work} expected={expected_control_work}"
    )
control_symlinks = [str(path) for path in control_work.rglob("*") if path.is_symlink()]
if control_symlinks:
    raise RuntimeError(f"Qwen control workdir contains symlinks: {control_symlinks[:20]}")
control_copy = AUTOLEARN_ROOT / "qwen_control_workdir"
if control_copy.exists():
    shutil.rmtree(control_copy)
shutil.copytree(control_work, control_copy, dirs_exist_ok=False)
def validated_evidence_file(path: Path, *, label: str) -> Path:
    if path.is_symlink() or not path.is_file():
        raise RuntimeError(f"required evidence is not a regular file: label={label} path={path}")
    resolved = path.resolve(strict=True)
    if resolved.stat().st_size <= 0:
        raise RuntimeError(f"required evidence is empty: label={label} path={resolved}")
    assert_no_sensitive_bytes(resolved.read_bytes(), context=f"final evidence {label}")
    return resolved


def create_sealed_evidence_zip(root: Path, destination: Path) -> dict:
    root = root.resolve(strict=True)
    manifest_path = root / "FINAL_EVIDENCE_MANIFEST.json"
    if manifest_path.exists():
        raise FileExistsError(f"final evidence manifest already exists: {manifest_path}")
    if destination.exists():
        raise FileExistsError(f"final evidence archive already exists: {destination}")
    files = sorted(path for path in root.rglob("*") if path.is_file())
    if not files:
        raise RuntimeError("final evidence root contains no files")
    records = []
    payloads = []
    for path in files:
        if path.is_symlink():
            raise RuntimeError(f"final evidence contains a symlink: {path}")
        resolved = path.resolve(strict=True)
        if root not in resolved.parents:
            raise RuntimeError(f"final evidence escaped its root: {resolved}")
        payload = resolved.read_bytes()
        assert_no_sensitive_bytes(payload, context=f"sealed evidence {resolved.name}")
        relative = resolved.relative_to(root).as_posix()
        records.append({
            "path": relative,
            "size_bytes": len(payload),
            "sha256": hashlib.sha256(payload).hexdigest(),
        })
        payloads.append((relative, payload))
    payload_names = [relative for relative, _ in payloads]
    if len(payload_names) != len(set(payload_names)):
        raise RuntimeError("final evidence payload contains duplicate archive names")
    manifest = {
        "schema": "arc-agi-2-p147-final-evidence-v1",
        "release": "p147-20260801",
        "run_id": AUTOLEARN_RUN_ID,
        "lab_parameters_sha256": LAB_PARAMETERS_SHA256,
        "embedded_bundle_sha256": EMBEDDED_BUNDLE_SHA256,
        "executable_source_sha256": P147_EXECUTABLE_SOURCE_SHA256,
        "builder_sha256": P147_BUILDER_SHA256,
        "episode_identity_sha256": EPISODE_IDENTITY["episode_identity_sha256"],
        "file_count": len(records),
        "files": records,
    }
    manifest_payload = (json.dumps(manifest, indent=2, sort_keys=True) + "\n").encode("utf-8")
    atomic_write_bytes(manifest_path, manifest_payload)
    if json.loads(manifest_path.read_text(encoding="utf-8", errors="strict")) != manifest:
        raise RuntimeError("final evidence manifest failed its read-back check")

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".tmp")
    temporary.unlink(missing_ok=True)
    try:
        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for relative, payload in payloads:
                info = zipfile.ZipInfo(relative, date_time=(1980, 1, 1, 0, 0, 0))
                info.compress_type = zipfile.ZIP_DEFLATED
                info.external_attr = 0o100600 << 16
                info.create_system = 3
                archive.writestr(info, payload)
            manifest_info = zipfile.ZipInfo(
                "FINAL_EVIDENCE_MANIFEST.json",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            manifest_info.compress_type = zipfile.ZIP_DEFLATED
            manifest_info.external_attr = 0o100600 << 16
            manifest_info.create_system = 3
            archive.writestr(manifest_info, manifest_payload)
        with zipfile.ZipFile(temporary) as archive:
            if archive.testzip() is not None:
                raise RuntimeError("final evidence ZIP failed CRC validation")
            expected_names = [relative for relative, _ in payloads] + [
                "FINAL_EVIDENCE_MANIFEST.json"
            ]
            if archive.namelist() != expected_names:
                raise RuntimeError("final evidence ZIP inventory mismatch")
            if archive.read("FINAL_EVIDENCE_MANIFEST.json") != manifest_payload:
                raise RuntimeError("final evidence ZIP manifest mismatch")
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)
    archive_payload = destination.read_bytes()
    assert_no_sensitive_bytes(archive_payload, context="final evidence archive")
    return {
        "manifest_path": str(manifest_path),
        "manifest_sha256": hashlib.sha256(manifest_payload).hexdigest(),
        "archive_path": str(destination),
        "archive_size_bytes": len(archive_payload),
        "archive_sha256": hashlib.sha256(archive_payload).hexdigest(),
        "archive_entry_count": len(payloads) + 1,
    }


runtime_provenance_dir = AUTOLEARN_ROOT / "runtime_provenance"
runtime_provenance_dir.mkdir(parents=True, exist_ok=False)
runtime_provenance_sources = {
    "dependency_contract.json": json.loads(
        DEPENDENCY_CONTRACT_PATH.read_text(encoding="utf-8", errors="strict")
    ),
    "installed_runtime_preflight.json": P147_INSTALLED_RUNTIME_PREFLIGHT,
    "staged_model_contract.json": stage_config,
    "compat_unsloth_runtime.json": COLAB_COMPAT_UNSLOTH_REPORT,
    "child_environment_policy.json": {
        "python_no_user_site": True,
        "blocked_exact": sorted(PACKAGE_ENV_EXACT_BLOCKLIST),
        "blocked_prefixes": ["PIP_"],
        "trusted_path": TRUSTED_EXECUTABLE_PATH,
    },
    "executable_identity.json": {
        "normalized_source_sha256": P147_EXECUTABLE_SOURCE_SHA256,
        "builder_sha256": P147_BUILDER_SHA256,
        "embedded_bundle_sha256": EMBEDDED_BUNDLE_SHA256,
    },
}
for provenance_name, provenance_payload in runtime_provenance_sources.items():
    atomic_write_text(
        runtime_provenance_dir / provenance_name,
        json.dumps(provenance_payload, indent=2, sort_keys=True),
    )

key_autolearn_sources = {
    "episode_manifest.json": EPISODE_MANIFEST_PATH,
    "candidate_manifest.json": CANDIDATE_MANIFEST_PATH,
    "autolearning_report.json": RANKER_DIR / "autolearning_report.json",
    "candidate_trace.jsonl": RANKER_DIR / "candidate_trace.jsonl",
    "metric_trace.jsonl": RANKER_DIR / "metric_trace.jsonl",
    "task_trace.jsonl": RANKER_DIR / "task_trace.jsonl",
    "selection_trace.jsonl": RANKER_DIR / "selection_trace.jsonl",
    "process_trace.jsonl": PROCESS_TRACE_PATH,
    "dependency_contract.json": runtime_provenance_dir / "dependency_contract.json",
    "installed_runtime_preflight.json": runtime_provenance_dir / "installed_runtime_preflight.json",
    "staged_model_contract.json": runtime_provenance_dir / "staged_model_contract.json",
    "compat_unsloth_runtime.json": runtime_provenance_dir / "compat_unsloth_runtime.json",
    "child_environment_policy.json": runtime_provenance_dir / "child_environment_policy.json",
    "executable_identity.json": runtime_provenance_dir / "executable_identity.json",
}
required_evidence_dir = AUTOLEARN_ROOT / "required_evidence"
if required_evidence_dir.exists():
    shutil.rmtree(required_evidence_dir)
required_evidence_dir.mkdir(parents=True, exist_ok=False)
key_autolearn_paths = []
for evidence_name, source_path in key_autolearn_sources.items():
    source_path = validated_evidence_file(Path(source_path), label=evidence_name)
    destination = required_evidence_dir / evidence_name
    atomic_write_bytes(destination, source_path.read_bytes())
    key_autolearn_paths.append(destination)

task_trace_rows = [
    json.loads(line)
    for line in (required_evidence_dir / "task_trace.jsonl").read_text(
        encoding="utf-8", errors="strict"
    ).splitlines()
    if line.strip()
]
task_trace_ids = [row.get("episode_id") for row in task_trace_rows if isinstance(row, dict)]
if len(task_trace_rows) != MAX_TASKS or len(set(task_trace_ids)) != MAX_TASKS:
    raise RuntimeError(
        f"task trace inventory mismatch: rows={len(task_trace_rows)} unique={len(set(task_trace_ids))} expected={MAX_TASKS}"
    )
trace_incomplete_rows = [
    row.get("episode_id")
    for row in task_trace_rows
    if "task_not_completed_in_trace" in row.get("secondary_flags", [])
]
if trace_incomplete_rows:
    raise RuntimeError(f"task trace contains incomplete episodes: {trace_incomplete_rows[:20]}")

LOPO_SOLUTIONS.unlink(missing_ok=True)
LOPO_SOLUTIONS_PAYLOAD = b""
archive_path = AUTOLEARN_ROOT.parent / f"{AUTOLEARN_RUN_ID}_complete_artifacts.zip"
if archive_path.exists():
    raise FileExistsError(f"final evidence archive path already exists: {archive_path}")
FINAL_EVIDENCE_RECEIPT = create_sealed_evidence_zip(AUTOLEARN_ROOT, archive_path)
receipt_path = AUTOLEARN_ROOT.parent / f"{AUTOLEARN_RUN_ID}_final_evidence_receipt.json"
atomic_write_text(receipt_path, json.dumps(FINAL_EVIDENCE_RECEIPT, indent=2, sort_keys=True))

autolearn_output_dir = out_dir / "autolearning"
autolearn_output_dir.mkdir(parents=True, exist_ok=True)
drive_archive = autolearn_output_dir / archive_path.name
drive_receipt = autolearn_output_dir / receipt_path.name
atomic_write_bytes(drive_archive, archive_path.read_bytes())
atomic_write_bytes(drive_receipt, receipt_path.read_bytes())
for source_path in key_autolearn_paths:
    atomic_write_bytes(autolearn_output_dir / source_path.name, source_path.read_bytes())
artifact_paths.extend([
    archive_path,
    receipt_path,
    drive_archive,
    drive_receipt,
    Path(FINAL_EVIDENCE_RECEIPT["manifest_path"]),
    *key_autolearn_paths,
])
print("autolearning archive", archive_path, archive_path.stat().st_size)
print("final evidence receipt", json.dumps(FINAL_EVIDENCE_RECEIPT, sort_keys=True))
success_finalizer_errors = []
if HF_BRIDGE is not None:
    try:
        HF_BRIDGE.stop(
            "complete",
            extra_paths=artifact_paths,
            extra={
                "output_dir": str(out_dir),
                "drive_available": DRIVE_AVAILABLE,
                "decision": decision_json,
                "generation_candidate_decision": generation_decision_json,
                "rows": rows,
                "lab_parameters": LAB_PARAMETERS,
                "final_evidence_receipt": FINAL_EVIDENCE_RECEIPT,
            },
        )
    except Exception as finalizer_exc:
        success_finalizer_errors.append(
            _redact_sensitive_text(
                f"hf_stop:{type(finalizer_exc).__name__}:{finalizer_exc}"
            )
        )
if DRIVE_LOG_MIRROR is not None:
    try:
        DRIVE_LOG_MIRROR.stop()
    except Exception as finalizer_exc:
        success_finalizer_errors.append(
            _redact_sensitive_text(
                f"drive_stop:{type(finalizer_exc).__name__}:{finalizer_exc}"
            )
        )
if success_finalizer_errors:
    raise RuntimeError(
        f"final log synchronization failed after all sinks were attempted: {success_finalizer_errors}"
    )
print("\nDONE. P147 completed hidden-parity autolearning calibration; it did not submit to Kaggle.")